# 05 — sakai224_full label-blind k-means++ diagnostics

Questo notebook valuta quantitativamente gli embedding deterministici del VAE
`sakai224_full` dopo 100 epoche.

Gli input scientifici sono gli embedding canonici:

$$
M_{\mu}^{\mathrm{train}}
\in
\mathbb{R}^{6866\times512},
$$

$$
M_{\mu}^{\mathrm{validation}}
\in
\mathbb{R}^{1717\times512}.
$$

Il protocollo è:

$$
M_{\mu}^{\mathrm{train}}
\rightarrow
\mathrm{k\text{-}means++}
\rightarrow
k\in\{22,36\}
\rightarrow
\text{centroidi train},
$$

seguito da:

$$
M_{\mu}^{\mathrm{validation}}
\rightarrow
\text{assegnazione ai centroidi train}.
$$

Il Notebook 05 è integralmente label-blind. Le label Gravity Spy non vengono
lette e non partecipano al fit, all'inizializzazione, alla scelta di
`k=22` o `k=36`, alla scelta fra `raw_mu` e `zscore_train_only` o alla
pubblicazione dell'artefatto.

I valori `k=22` e `k=36` sono pre-registrati rispettivamente come controllo
della cardinalità delle classi e come cardinalità nominale Sakai; non sono
selezionati a posteriori tramite label.

Nel Notebook 05 vengono eseguiti:

- caricamento delle μ512 canoniche;
- clustering k-means label-blind con `k=22` e `k=36`;
- analisi primaria `raw_mu` e sensitivity analysis `zscore_train_only`;
- fit esclusivamente sul train e assegnazione della validation ai centroidi
  train congelati;
- diagnostiche interne unsupervised;
- stabilità fra le partizioni raw e z-score mediante ARI e NMI.

ARI e NMI raw-vs-z-score confrontano due partizioni e non sono metriche
esterne rispetto alle label Gravity Spy.

Nel Notebook 05 non vengono eseguiti:

- matrici di contingenza Gravity Spy;
- purity;
- NMI o ARI rispetto alle label;
- homogeneity;
- completeness;
- altre valutazioni label-aware.

Queste analisi appartengono a una fase post-hoc separata, successiva al
congelamento delle partizioni.

Il ramo Python costituisce la valutazione nativa del modello
`sakai224_full`.

Il confronto degli stessi embedding tramite il backend C comune è separato
dal presente notebook e usa la stessa implementazione di clustering e metriche:

$$
\{
\mathrm{mini75},
\mathrm{mini112},
\mathrm{mini224\_1000},
\mathrm{sakai224\_full}
\}.
$$

## STEP 05-A0 — Audit degli embedding pubblicati e inventario KMeans

Prima di definire il clustering vengono riletti esclusivamente gli artefatti
canonici pubblicati dal Notebook 04.

Il preflight verifica:

- integrità SHA-256 dei file;
- formato `float32` little-endian row-major;
- shape train e validation;
- identità e ordine delle row-map;
- assenza delle label dall'artefatto;
- copertura completa e disgiunzione degli split;
- versione delle librerie numeriche;
- firma effettiva dell'implementazione `KMeans`.

Nessun default del software viene accettato implicitamente.

Il contratto congela, prima del fit:

$$
\mathrm{init},
\qquad
n_{\mathrm{init}},
\qquad
s_{\mathrm{random}},
\qquad
n_{\mathrm{iter,max}},
\qquad
\mathrm{tol},
\qquad
\mathrm{algorithm},
$$

e il trattamento delle feature.

Il preflight non esegue alcun clustering e non legge le label Gravity Spy.

In [1]:
# ============================================================
# NOTEBOOK 05 — EMBEDDING / KMEANS API PREFLIGHT
# STEP 05-A0
#
# READ ONLY.
#
# Verifies the canonical epoch-100 mu artifact and inventories
# the current sklearn KMeans implementation.
#
# NO clustering.
# NO labels.
# NO persistent writes.
# ============================================================

from pathlib import Path

import csv
import hashlib
import importlib.util
import inspect
import json
import os
import platform
import tempfile

import numpy as np
import scipy
import sklearn
import threadpoolctl

from sklearn.cluster import KMeans
from threadpoolctl import threadpool_info


# ============================================================
# PORTABLE ROOT CONTRACT
# ============================================================

PROJECT_ROOT_ENV = os.environ.get(
    "GRAVITYSPY_PROJECT_ROOT"
)

LOCAL_RUNTIME_ROOT_ENV = os.environ.get(
    "GRAVITYSPY_LOCAL_RUNTIME_ROOT"
)


try:
    IN_COLAB = (
        importlib.util.find_spec(
            "google.colab"
        )
        is not None
    )
except ModuleNotFoundError:
    IN_COLAB = False


if LOCAL_RUNTIME_ROOT_ENV is not None:

    LOCAL_RUNTIME_ROOT = Path(
        LOCAL_RUNTIME_ROOT_ENV
    )

    if not LOCAL_RUNTIME_ROOT.is_absolute():

        raise ValueError(
            "GRAVITYSPY_LOCAL_RUNTIME_ROOT must be an absolute path."
        )

elif IN_COLAB:

    LOCAL_RUNTIME_ROOT = Path(
        "/content"
    )

else:

    LOCAL_RUNTIME_ROOT = Path(
        tempfile.gettempdir()
    )


if not LOCAL_RUNTIME_ROOT.is_absolute():

    raise RuntimeError(
        "LOCAL_RUNTIME_ROOT must resolve to an absolute path."
    )


if PROJECT_ROOT_ENV is not None:

    PROJECT_ROOT = Path(
        PROJECT_ROOT_ENV
    )

    if not PROJECT_ROOT.is_absolute():

        raise ValueError(
            "GRAVITYSPY_PROJECT_ROOT must be an absolute path."
        )

elif IN_COLAB:

    DRIVE_MOUNT_POINT = Path(
        "/content/drive"
    )

    if not (
        DRIVE_MOUNT_POINT
        / "MyDrive"
    ).is_dir():

        from google.colab import drive

        drive.mount(
            str(
                DRIVE_MOUNT_POINT
            )
        )

    PROJECT_ROOT = (
        DRIVE_MOUNT_POINT
        / "MyDrive"
        / "TESI_SAKAI2022"
    )

else:

    raise RuntimeError(
        "GRAVITYSPY_PROJECT_ROOT must be set to an absolute path outside Colab."
    )


# ============================================================
# PERSISTENT PATHS
# ============================================================

SAKAI224_ROOT = (
    PROJECT_ROOT
    / "sakai224_full"
)

CANONICAL_EMBEDDING_DIR = (
    SAKAI224_ROOT
    / "embeddings"
    / "sakai224_full_mu_epoch100_v1"
)


TRAIN_BIN_NAME = (
    "embeddings_mu_train_f32.bin"
)

VALIDATION_BIN_NAME = (
    "embeddings_mu_validation_f32.bin"
)

TRAIN_ROW_MAP_NAME = (
    "row_map_train.csv"
)

VALIDATION_ROW_MAP_NAME = (
    "row_map_validation.csv"
)

METADATA_NAME = (
    "metadata.json"
)

SHA256SUMS_NAME = (
    "SHA256SUMS.txt"
)


TRAIN_BIN_PATH = (
    CANONICAL_EMBEDDING_DIR
    / TRAIN_BIN_NAME
)

VALIDATION_BIN_PATH = (
    CANONICAL_EMBEDDING_DIR
    / VALIDATION_BIN_NAME
)

TRAIN_ROW_MAP_PATH = (
    CANONICAL_EMBEDDING_DIR
    / TRAIN_ROW_MAP_NAME
)

VALIDATION_ROW_MAP_PATH = (
    CANONICAL_EMBEDDING_DIR
    / VALIDATION_ROW_MAP_NAME
)

METADATA_PATH = (
    CANONICAL_EMBEDDING_DIR
    / METADATA_NAME
)

SHA256SUMS_PATH = (
    CANONICAL_EMBEDDING_DIR
    / SHA256SUMS_NAME
)


# ============================================================
# FROZEN IDENTITIES
# ============================================================

EXPECTED_ARTIFACT_ID = (
    "sakai224_full_mu_epoch100_v1"
)

EXPECTED_TRAIN_ROWS = 6866
EXPECTED_VALIDATION_ROWS = 1717
EXPECTED_LATENT_DIM = 512

EXPECTED_TRAIN_BYTES = (
    EXPECTED_TRAIN_ROWS
    * EXPECTED_LATENT_DIM
    * 4
)

EXPECTED_VALIDATION_BYTES = (
    EXPECTED_VALIDATION_ROWS
    * EXPECTED_LATENT_DIM
    * 4
)


EXPECTED_TRAIN_MU_SHA256 = (
    "b620fd9711719a7bd40293b6536ea6fe"
    "b235cbadb1d1e37dca0e46651332ad5e"
)

EXPECTED_VALIDATION_MU_SHA256 = (
    "c1692b3e7a4c86a7f4d6d7172117e110"
    "83d2bd2394947838a24d1e2b81f27cf9"
)

EXPECTED_SHA256SUMS_SHA256 = (
    "2d6205547e6a063b49178595de9a24918"
    "bd0add1a4586c800e7519bf842f951d"
)

EXPECTED_FINAL_FILE_SHA256 = (
    "2bb6fd2ce2f73f8f9e479ed6969a52c1"
    "cdd41c1ae6e2cfc33da687ebba798f71"
)

EXPECTED_FINAL_MODEL_STATE_SHA256 = (
    "3ac5a1cecdee2aa204419290b5cec55f"
    "3e6a4a846beebaa3f6b4e495f6fe6096"
)

EXPECTED_TENSOR_MANIFEST_SHA256 = (
    "94b7e2e051bd32de72dd2cb6a5fb7eda"
    "9bcaec123cbf3a07d0d901bd19779afa"
)

EXPECTED_TRAIN_SPLIT_SHA256 = (
    "8c7a513b98dbe996261bc80965500ab15"
    "ab77f498ec9440858003152e42f926f"
)

EXPECTED_VALIDATION_SPLIT_SHA256 = (
    "e63109c88df7b3ee97317f94791f26df"
    "3fcaf8b937921cfb27967d01c2546467"
)


EXPECTED_ARTIFACT_FILES = sorted([
    TRAIN_BIN_NAME,
    VALIDATION_BIN_NAME,
    TRAIN_ROW_MAP_NAME,
    VALIDATION_ROW_MAP_NAME,
    METADATA_NAME,
    SHA256SUMS_NAME,
])


# ============================================================
# STREAMING SHA-256
# ============================================================

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):

    digest = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as handle:

        while True:

            block = handle.read(
                chunk_size
            )

            if not block:
                break

            digest.update(
                block
            )

    return digest.hexdigest()


# ============================================================
# DIRECTORY CONTRACT
# ============================================================

assert CANONICAL_EMBEDDING_DIR.is_dir()


actual_artifact_files = sorted(
    path.name

    for path
    in CANONICAL_EMBEDDING_DIR.iterdir()

    if path.is_file()
)


directory_gate = (
    actual_artifact_files
    == EXPECTED_ARTIFACT_FILES
)


assert directory_gate


# ============================================================
# SHA256SUMS PARSER
# ============================================================

def read_sha256sums(
    path,
):

    records = {}


    with Path(path).open(
        "r",
        encoding="ascii",
    ) as handle:

        for line_number, line in enumerate(
            handle,
            start=1,
        ):

            line = line.strip()

            if not line:
                continue


            parts = line.split(
                maxsplit=1
            )


            if len(parts) != 2:

                raise RuntimeError(
                    "Malformed SHA256SUMS line "
                    f"{line_number}"
                )


            digest = parts[0]
            filename = parts[1].strip()


            if (
                len(digest) != 64
                or filename in records
            ):

                raise RuntimeError(
                    "Invalid SHA256SUMS entry "
                    f"at line {line_number}"
                )


            records[
                filename
            ] = digest


    return records


listed_hashes = read_sha256sums(
    SHA256SUMS_PATH
)


expected_listed_files = set([
    TRAIN_BIN_NAME,
    VALIDATION_BIN_NAME,
    TRAIN_ROW_MAP_NAME,
    VALIDATION_ROW_MAP_NAME,
    METADATA_NAME,
])


sha_manifest_gate = (
    sha256_file(
        SHA256SUMS_PATH
    )
    == EXPECTED_SHA256SUMS_SHA256

    and set(
        listed_hashes
    )
    == expected_listed_files
)


assert sha_manifest_gate


# ============================================================
# VERIFY EVERY FILE LISTED IN SHA256SUMS
# ============================================================

calculated_hashes = {
    filename:
        sha256_file(
            CANONICAL_EMBEDDING_DIR
            / filename
        )

    for filename
    in sorted(
        listed_hashes
    )
}


all_file_hashes_gate = all(
    calculated_hashes[
        filename
    ]
    == listed_hashes[
        filename
    ]

    for filename
    in listed_hashes
)


binary_frozen_hash_gate = (
    calculated_hashes[
        TRAIN_BIN_NAME
    ]
    == EXPECTED_TRAIN_MU_SHA256

    and calculated_hashes[
        VALIDATION_BIN_NAME
    ]
    == EXPECTED_VALIDATION_MU_SHA256
)


assert all_file_hashes_gate
assert binary_frozen_hash_gate


# ============================================================
# METADATA CONTRACT
# ============================================================

with METADATA_PATH.open(
    "r",
    encoding="utf-8",
) as handle:

    embedding_metadata = json.load(
        handle
    )


metadata_gate = (
    embedding_metadata[
        "artifact_format_version"
    ] == 1

    and embedding_metadata[
        "artifact_id"
    ] == EXPECTED_ARTIFACT_ID

    and embedding_metadata[
        "experiment"
    ] == "sakai224_full"

    and embedding_metadata[
        "source_run_id"
    ] == "sakai224_full_v1"

    and embedding_metadata[
        "source_checkpoint"
    ] == "final.pt"

    and embedding_metadata[
        "source_completed_epochs"
    ] == 100

    and embedding_metadata[
        "source_last_completed_epoch"
    ] == 99

    and embedding_metadata[
        "source_optimizer_step"
    ] == 5400

    and embedding_metadata[
        "source_final_file_sha256"
    ] == EXPECTED_FINAL_FILE_SHA256

    and embedding_metadata[
        "source_model_state_sha256"
    ] == EXPECTED_FINAL_MODEL_STATE_SHA256

    and embedding_metadata[
        "canonical_feature"
    ] == "mu"

    and embedding_metadata[
        "latent_dim"
    ] == EXPECTED_LATENT_DIM

    and embedding_metadata[
        "dtype"
    ] == "float32"

    and embedding_metadata[
        "endianness"
    ] == "little"

    and embedding_metadata[
        "matrix_layout"
    ] == "row_major"

    and embedding_metadata[
        "values_per_row"
    ] == EXPECTED_LATENT_DIM

    and embedding_metadata[
        "train_rows"
    ] == EXPECTED_TRAIN_ROWS

    and embedding_metadata[
        "validation_rows"
    ] == EXPECTED_VALIDATION_ROWS

    and embedding_metadata[
        "pca_applied"
    ] is False

    and embedding_metadata[
        "standardization_applied"
    ] is False

    and embedding_metadata[
        "labels_in_embedding_artifact"
    ] is False

    and embedding_metadata[
        "train_mu_sha256"
    ] == EXPECTED_TRAIN_MU_SHA256

    and embedding_metadata[
        "validation_mu_sha256"
    ] == EXPECTED_VALIDATION_MU_SHA256

    and embedding_metadata[
        "manifest_tensors_224_central_sha256"
    ] == EXPECTED_TENSOR_MANIFEST_SHA256

    and embedding_metadata[
        "split_train_seed123_sha256"
    ] == EXPECTED_TRAIN_SPLIT_SHA256

    and embedding_metadata[
        "split_validation_seed123_sha256"
    ] == EXPECTED_VALIDATION_SPLIT_SHA256
)


assert metadata_gate


# ============================================================
# BINARY SIZE CONTRACT
# ============================================================

binary_size_gate = (
    TRAIN_BIN_PATH.stat().st_size
    == EXPECTED_TRAIN_BYTES

    and VALIDATION_BIN_PATH.stat().st_size
    == EXPECTED_VALIDATION_BYTES
)


assert binary_size_gate


# ============================================================
# READ-ONLY MEMORY MAPS
# ============================================================

train_mu_published = np.memmap(
    TRAIN_BIN_PATH,
    dtype="<f4",
    mode="r",
    shape=(
        EXPECTED_TRAIN_ROWS,
        EXPECTED_LATENT_DIM,
    ),
    order="C",
)


validation_mu_published = np.memmap(
    VALIDATION_BIN_PATH,
    dtype="<f4",
    mode="r",
    shape=(
        EXPECTED_VALIDATION_ROWS,
        EXPECTED_LATENT_DIM,
    ),
    order="C",
)


binary_numeric_gate = (
    train_mu_published.shape
    == (
        6866,
        512,
    )

    and validation_mu_published.shape
    == (
        1717,
        512,
    )

    and train_mu_published.dtype
    == np.dtype("<f4")

    and validation_mu_published.dtype
    == np.dtype("<f4")

    and train_mu_published.flags[
        "C_CONTIGUOUS"
    ]

    and validation_mu_published.flags[
        "C_CONTIGUOUS"
    ]

    and np.isfinite(
        train_mu_published
    ).all()

    and np.isfinite(
        validation_mu_published
    ).all()
)


assert binary_numeric_gate


# ============================================================
# ROW MAPS
# ============================================================

ROW_MAP_FIELDS = [
    "split",
    "split_pos",
    "manifest_index",
    "event_index",
    "event_id",
]


def read_row_map(
    path,
):

    with Path(path).open(
        "r",
        newline="",
        encoding="utf-8",
    ) as handle:

        reader = csv.DictReader(
            handle
        )

        rows = list(
            reader
        )

        header = list(
            reader.fieldnames
        )


    return header, rows


(
    train_row_map_header,
    train_row_map,
) = read_row_map(
    TRAIN_ROW_MAP_PATH
)


(
    validation_row_map_header,
    validation_row_map,
) = read_row_map(
    VALIDATION_ROW_MAP_PATH
)


row_map_header_gate = (
    train_row_map_header
    == ROW_MAP_FIELDS

    and validation_row_map_header
    == ROW_MAP_FIELDS

    and "label"
    not in train_row_map_header

    and "label"
    not in validation_row_map_header
)


assert row_map_header_gate


# ============================================================
# ROW MAP CONTENT
# ============================================================

def audit_row_map(
    rows,
    *,
    expected_split,
    expected_count,
):

    manifest_indices = []
    event_ids = []


    for expected_position, row in enumerate(
        rows
    ):

        manifest_index = int(
            row[
                "manifest_index"
            ]
        )

        event_index = int(
            row[
                "event_index"
            ]
        )


        if not (
            row[
                "split"
            ] == expected_split

            and int(
                row[
                    "split_pos"
                ]
            ) == expected_position

            and event_index
            == manifest_index + 1

            and row[
                "event_id"
            ]
        ):

            raise RuntimeError(
                "Row-map contract failure at "
                f"{expected_split} position "
                f"{expected_position}"
            )


        manifest_indices.append(
            manifest_index
        )

        event_ids.append(
            row[
                "event_id"
            ]
        )


    return {
        "manifest_indices":
            manifest_indices,

        "event_ids":
            event_ids,

        "count_gate":
            len(rows)
            == expected_count,

        "unique_manifest_gate":
            len(
                set(
                    manifest_indices
                )
            )
            == expected_count,

        "unique_event_id_gate":
            len(
                set(
                    event_ids
                )
            )
            == expected_count,
    }


train_row_map_audit = audit_row_map(
    train_row_map,
    expected_split="train",
    expected_count=6866,
)


validation_row_map_audit = audit_row_map(
    validation_row_map,
    expected_split="validation",
    expected_count=1717,
)


train_manifest_set = set(
    train_row_map_audit[
        "manifest_indices"
    ]
)

validation_manifest_set = set(
    validation_row_map_audit[
        "manifest_indices"
    ]
)


partition_gate = (
    train_row_map_audit[
        "count_gate"
    ]

    and train_row_map_audit[
        "unique_manifest_gate"
    ]

    and train_row_map_audit[
        "unique_event_id_gate"
    ]

    and validation_row_map_audit[
        "count_gate"
    ]

    and validation_row_map_audit[
        "unique_manifest_gate"
    ]

    and validation_row_map_audit[
        "unique_event_id_gate"
    ]

    and train_manifest_set.isdisjoint(
        validation_manifest_set
    )

    and (
        train_manifest_set
        | validation_manifest_set
    )
    == set(
        range(
            8583
        )
    )

    and len(
        set(
            train_row_map_audit[
                "event_ids"
            ]
        )
        |
        set(
            validation_row_map_audit[
                "event_ids"
            ]
        )
    )
    == 8583
)


assert partition_gate


prefix_gate = (
    train_row_map_audit[
        "manifest_indices"
    ][:4]
    == [
        7663,
        2606,
        7841,
        4067,
    ]

    and validation_row_map_audit[
        "manifest_indices"
    ][:4]
    == [
        6864,
        1488,
        8384,
        6585,
    ]

    and train_row_map_audit[
        "event_ids"
    ][0]
    == "H1_BqniZ6oLFl"

    and validation_row_map_audit[
        "event_ids"
    ][0]
    == "H1_4D7IcowE8a"
)


assert prefix_gate


# ============================================================
# KMEANS API INVENTORY
#
# No estimator is fitted or executed.
# ============================================================

kmeans_signature = inspect.signature(
    KMeans
)


kmeans_parameters = (
    kmeans_signature.parameters
)


required_kmeans_parameters = {
    "n_clusters",
    "init",
    "n_init",
    "max_iter",
    "tol",
    "verbose",
    "random_state",
    "copy_x",
    "algorithm",
}


kmeans_api_gate = (
    required_kmeans_parameters.issubset(
        set(
            kmeans_parameters
        )
    )
)


assert kmeans_api_gate


def parameter_default(
    name,
):

    return kmeans_parameters[
        name
    ].default


# ============================================================
# NUMERIC RUNTIME INVENTORY
# ============================================================

threadpool_inventory = threadpool_info()


# ============================================================
# REPORT
# ============================================================

print()
print(
    "=== NOTEBOOK 05 EMBEDDING / KMEANS PREFLIGHT ==="
)

print()

print(
    "ARTIFACT_DIRECTORY =",
    CANONICAL_EMBEDDING_DIR,
)

print(
    "ARTIFACT_FILES =",
    actual_artifact_files,
)

print()

print(
    "SHA256SUMS_SHA256 =",
    sha256_file(
        SHA256SUMS_PATH
    ),
)

print(
    "TRAIN_MU_SHA256 =",
    calculated_hashes[
        TRAIN_BIN_NAME
    ],
)

print(
    "VALIDATION_MU_SHA256 =",
    calculated_hashes[
        VALIDATION_BIN_NAME
    ],
)

print(
    "TRAIN_ROW_MAP_SHA256 =",
    calculated_hashes[
        TRAIN_ROW_MAP_NAME
    ],
)

print(
    "VALIDATION_ROW_MAP_SHA256 =",
    calculated_hashes[
        VALIDATION_ROW_MAP_NAME
    ],
)

print(
    "METADATA_SHA256 =",
    calculated_hashes[
        METADATA_NAME
    ],
)

print()

print(
    "TRAIN_MU_SHAPE =",
    train_mu_published.shape,
)

print(
    "VALIDATION_MU_SHAPE =",
    validation_mu_published.shape,
)

print(
    "DTYPE =",
    train_mu_published.dtype,
)

print(
    "LAYOUT = row_major"
)

print()

print(
    "TRAIN_FIRST_MANIFESTS =",
    train_row_map_audit[
        "manifest_indices"
    ][:4],
)

print(
    "VALIDATION_FIRST_MANIFESTS =",
    validation_row_map_audit[
        "manifest_indices"
    ][:4],
)

print()

print(
    "PYTHON_VERSION =",
    platform.python_version(),
)

print(
    "NUMPY_VERSION =",
    np.__version__,
)

print(
    "SCIPY_VERSION =",
    scipy.__version__,
)

print(
    "SCIKIT_LEARN_VERSION =",
    sklearn.__version__,
)

print(
    "THREADPOOLCTL_VERSION =",
    threadpoolctl.__version__,
)

print()

print(
    "KMEANS_SIGNATURE =",
    kmeans_signature,
)

print()

print(
    "KMEANS_DEFAULT_INIT =",
    parameter_default(
        "init"
    ),
)

print(
    "KMEANS_DEFAULT_N_INIT =",
    parameter_default(
        "n_init"
    ),
)

print(
    "KMEANS_DEFAULT_MAX_ITER =",
    parameter_default(
        "max_iter"
    ),
)

print(
    "KMEANS_DEFAULT_TOL =",
    parameter_default(
        "tol"
    ),
)

print(
    "KMEANS_DEFAULT_RANDOM_STATE =",
    parameter_default(
        "random_state"
    ),
)

print(
    "KMEANS_DEFAULT_COPY_X =",
    parameter_default(
        "copy_x"
    ),
)

print(
    "KMEANS_DEFAULT_ALGORITHM =",
    parameter_default(
        "algorithm"
    ),
)

print()

print(
    "THREADPOOL_INVENTORY =",
    threadpool_inventory,
)

print()

print(
    "DIRECTORY_GATE =",
    directory_gate,
)

print(
    "SHA_MANIFEST_GATE =",
    sha_manifest_gate,
)

print(
    "ALL_FILE_HASHES_GATE =",
    all_file_hashes_gate,
)

print(
    "BINARY_FROZEN_HASH_GATE =",
    binary_frozen_hash_gate,
)

print(
    "METADATA_GATE =",
    metadata_gate,
)

print(
    "BINARY_SIZE_GATE =",
    binary_size_gate,
)

print(
    "BINARY_NUMERIC_GATE =",
    binary_numeric_gate,
)

print(
    "ROW_MAP_HEADER_GATE =",
    row_map_header_gate,
)

print(
    "PARTITION_GATE =",
    partition_gate,
)

print(
    "PREFIX_GATE =",
    prefix_gate,
)

print(
    "KMEANS_API_GATE =",
    kmeans_api_gate,
)

print()

print(
    "LABELS_READ =",
    False,
)

print(
    "KMEANS_EXECUTED =",
    False,
)

print(
    "CLUSTER_ASSIGNMENTS_CREATED =",
    False,
)

print(
    "OFFICIAL_DRIVE_ARTIFACTS_MODIFIED =",
    False,
)

print()

print(
    "NOTEBOOK_05_A0_PREFLIGHT_PASS =",
    (
        directory_gate
        and sha_manifest_gate
        and all_file_hashes_gate
        and binary_frozen_hash_gate
        and metadata_gate
        and binary_size_gate
        and binary_numeric_gate
        and row_map_header_gate
        and partition_gate
        and prefix_gate
        and kmeans_api_gate
    ),
)

Mounted at /content/drive

=== NOTEBOOK 05 EMBEDDING / KMEANS PREFLIGHT ===

ARTIFACT_DIRECTORY = /content/drive/MyDrive/TESI_SAKAI2022/sakai224_full/embeddings/sakai224_full_mu_epoch100_v1
ARTIFACT_FILES = ['SHA256SUMS.txt', 'embeddings_mu_train_f32.bin', 'embeddings_mu_validation_f32.bin', 'metadata.json', 'row_map_train.csv', 'row_map_validation.csv']

SHA256SUMS_SHA256 = 2d6205547e6a063b49178595de9a24918bd0add1a4586c800e7519bf842f951d
TRAIN_MU_SHA256 = b620fd9711719a7bd40293b6536ea6feb235cbadb1d1e37dca0e46651332ad5e
VALIDATION_MU_SHA256 = c1692b3e7a4c86a7f4d6d7172117e11083d2bd2394947838a24d1e2b81f27cf9
TRAIN_ROW_MAP_SHA256 = 4b25c7f41a945fc8195e93c11e60da3b2e5b8be8298f6919008fc57dc39cc9e8
VALIDATION_ROW_MAP_SHA256 = f5a0e48c79db4d466f834cbc54bdbf7e8c34d3fa45f545f58b37c71627c97b48
METADATA_SHA256 = 075883af82276431269e57a064da7fcdbd40e402fe8d58b8b1a3c63250a8c1e1

TRAIN_MU_SHAPE = (6866, 512)
VALIDATION_MU_SHAPE = (1717, 512)
DTYPE = float32
LAYOUT = row_major

TRAIN_FIRST_MANIFESTS 

## STEP 05-A1 — Contratto PROJECT del k-means per la valutazione del VAE full

L'obiettivo principale del progetto è confrontare `sakai224_full` con la
pipeline pubblicata da Sakai et al.

Il k-means di questo notebook **non è una replica della classificazione
finale di Sakai**.

La pipeline principale pubblicata utilizza infatti:

$$
\mathrm{VAE}
\longrightarrow
\mathrm{IIC}
\longrightarrow
K\text{ classificatori}
\longrightarrow
\mathrm{spectral\ clustering}.
$$

Il k-means viene introdotto come diagnostica PROJECT separata per rispondere
alla domanda:

> Le feature apprese dal nostro VAE full possiedono già una struttura
> clusterizzabile mediante un algoritmo euclideo semplice?

Il risultato del k-means non verrà quindi confrontato direttamente con
l'accuracy finale pubblicata da Sakai.

---

### Feature utilizzate

La rappresentazione canonica rimane la media dell'encoder:

$$
\mu_\phi(x)\in\mathbb{R}^{512}.
$$

L'analisi primaria utilizza direttamente:

$$
X_{\mathrm{train}}
=
M_{\mu}^{\mathrm{train}}
\in
\mathbb{R}^{6866\times512},
$$

e

$$
X_{\mathrm{validation}}
=
M_{\mu}^{\mathrm{validation}}
\in
\mathbb{R}^{1717\times512}.
$$

Non vengono applicate trasformazioni:

$$
\mathrm{PCA}=\mathrm{False},
$$

$$
\mathrm{standardizzazione}=\mathrm{False}.
$$

La scelta `raw_mu` è primaria perché mantiene inalterata la geometria
prodotta dall'encoder VAE.

---

### Sensitivity analysis pre-registrata

Il k-means è sensibile alla scala delle singole coordinate.

Per questo viene pre-registrata una seconda analisi, separata dai risultati
primari, con standardizzazione train-only:

$$
\tilde{\mu}_{i,j}
=
\frac{
\mu_{i,j}-\bar{\mu}^{\mathrm{train}}_j
}{
s_j^{\mathrm{train}}
}.
$$

La media viene calcolata esclusivamente sul training:

$$
\bar{\mu}^{\mathrm{train}}_j
=
\frac{1}{N_{\mathrm{train}}}
\sum_{i=1}^{N_{\mathrm{train}}}
\mu_{i,j},
$$

e la varianza population è:

$$
\left(s_j^{\mathrm{train}}\right)^2
=
\frac{1}{N_{\mathrm{train}}}
\sum_{i=1}^{N_{\mathrm{train}}}
\left(
\mu_{i,j}
-
\bar{\mu}^{\mathrm{train}}_j
\right)^2.
$$

Gli stessi parametri vengono poi applicati alla validation senza refit.

La variante standardizzata non sostituisce il risultato primario e non viene
scelta sulla base delle label.

---

### Numero di cluster

Vengono studiati due valori pre-registrati:

$$
k=22,
$$

e

$$
k=36.
$$

Il valore

$$
k=22
$$

è utilizzato come controllo rispetto al numero delle classi Gravity Spy note.

Il valore

$$
k=36
$$

è particolarmente rilevante perché la configurazione principale di Sakai
utilizza:

$$
C=36
$$

come numero di classi nell'IIC.

L'uso di $k=36$ nel k-means non implica che il k-means replichi l'IIC:
serve soltanto a confrontare due partizionamenti con lo stesso numero nominale
di gruppi.

---

### Obiettivo k-means

Per i centroidi

$$
C=
\{c_1,\ldots,c_k\},
$$

il fit minimizza sul training:

$$
J(C)
=
\sum_{i=1}^{N_{\mathrm{train}}}
\min_{j\in\{1,\ldots,k\}}
\left\|
x_i-c_j
\right\|_2^2.
$$

L'inizializzazione utilizza k-means++.

---

### Scelte PROJECT dell'implementazione Python

Il protocollo principale viene congelato come:

$$
\mathrm{init}
=
\texttt{k-means++},
$$

$$
n_{\mathrm{init}}
=
10,
$$

$$
s_{\mathrm{random}}
=
123,
$$

$$
n_{\mathrm{iter,max}}
=
300,
$$

$$
\mathrm{tol}
=
10^{-4},
$$

con:

`algorithm = "lloyd"`.

Questi valori non sono attribuiti a Sakai et al.

---

### Perché $n_{\mathrm{init}}=10$

Il k-means è sensibile all'inizializzazione dei centroidi.

Una singola inizializzazione potrebbe produrre accidentalmente un minimo
locale mediocre e dare un giudizio ingiustamente negativo sulla qualità
delle feature VAE.

Vengono quindi eseguite dieci inizializzazioni k-means++:

$$
J_1,J_2,\ldots,J_{10},
$$

e l'implementazione mantiene la soluzione con inerzia minima:

$$
J^\star
=
\min_{r\in\{1,\ldots,10\}}
J_r.
$$

Questa scelta rende la diagnostica delle feature più robusta rispetto
all'inizializzazione.

---

### Perché `random_state = 123`

Il valore

$$
s_{\mathrm{random}}=123
$$

è una scelta PROJECT esclusivamente di riproducibilità.

Non deriva dal paper e, nel ramo Python principale, non viene scelto per
replicare il backend C.

Una volta congelato, impedisce che successive esecuzioni cambino le dieci
inizializzazioni pseudo-casuali.

---

### Perché $\mathrm{tol}=10^{-4}$

Il paper non pubblica un criterio di convergenza k-means.

Viene mantenuta la normale semantica numerica dell'implementazione Python:

$$
\mathrm{tol}=10^{-4}.
$$

Non viene imposto `tol = 0`, perché non esiste un requisito PAPER di
convergenza bitwise o esatta e tale scelta sarebbe stata motivata
principalmente dal confronto con il backend C.

---

### Fit esclusivamente sul train

Il modello viene fit soltanto su:

$$
X_{\mathrm{train}}.
$$

Dopo il fit, i centroidi vengono congelati.

Gli eventi validation vengono assegnati tramite:

$$
a_i^{\mathrm{validation}}
=
\operatorname*{argmin}_{j}
\left\|
x_i^{\mathrm{validation}}
-
c_j
\right\|_2^2.
$$

Non viene effettuato alcun refit sulla validation.

---

### Utilizzo delle label

Le label Gravity Spy non partecipano:

- alle feature;
- all'inizializzazione;
- al fit;
- alla scelta della migliore inizializzazione;
- alla convergenza;
- all'assegnazione della validation.

Le label verranno associate agli assignment soltanto dopo il congelamento
del clustering.

---

### Diagnostiche interne ed esterne

Prima dell'uso delle label vengono calcolate metriche interne:

$$
\mathrm{Inertia},
$$

$$
\mathrm{Silhouette},
$$

$$
\mathrm{Davies\text{-}Bouldin}.
$$

Successivamente, a posteriori, verranno calcolate:

$$
\mathrm{Purity},
\qquad
\mathrm{NMI},
\qquad
\mathrm{ARI},
$$

$$
\mathrm{Homogeneity},
\qquad
\mathrm{Completeness}.
$$

Le due famiglie di metriche vengono mantenute separate.

---

### Rapporto con il confronto dei quattro VAE

Il confronto fra

$$
\{
\mathrm{mini75},
\mathrm{mini112},
\mathrm{mini224\_1000},
\mathrm{sakai224\_full}
\}
$$

è un esperimento successivo e distinto.

In quel confronto verrà utilizzato un unico backend C comune.

Le scelte del presente k-means Python non sono determinate dal backend C e
non vengono utilizzate per dichiarare superiorità fra i quattro VAE.

In [3]:
# ============================================================
# NOTEBOOK 05 — FREEZE FULL-VAE KMEANS PROJECT CONTRACT
# STEP 05-A1 — corrected / superseding version
#
# PRIMARY GOAL:
#   evaluate sakai224_full in its native Python branch.
#
# The common C comparison is explicitly separate.
#
# NO clustering.
# NO labels.
# NO Drive writes.
# ============================================================

import hashlib
import json

import numpy as np
import sklearn


# ============================================================
# SUPERSEDED CONTRACT
#
# The previous configuration was frozen before clarifying that
# the main scientific goal is sakai224_full vs Sakai.
#
# It produced NO clusters and NO persistent artifact.
# ============================================================

SUPERSEDED_KMEANS_CONFIG_SHA256 = (
    "f435f43f7b3afdb813d7b7e9b3c15c2c"
    "f84741bba7619ab7d5c8d6cd8349e2ec"
)


# ============================================================
# INPUT IDENTITY
# ============================================================

assert train_mu_published.shape == (
    6866,
    512,
)

assert validation_mu_published.shape == (
    1717,
    512,
)

assert calculated_hashes[
    TRAIN_BIN_NAME
] == EXPECTED_TRAIN_MU_SHA256

assert calculated_hashes[
    VALIDATION_BIN_NAME
] == EXPECTED_VALIDATION_MU_SHA256


# ============================================================
# CORRECTED FROZEN CONTRACT
# ============================================================

KMEANS_CONFIG = {

    "config_version":
        2,

    "supersedes_config_sha256":
        SUPERSEDED_KMEANS_CONFIG_SHA256,

    "scientific_role":
        (
            "PROJECT diagnostic evaluation of the "
            "sakai224_full VAE latent representation"
        ),

    "main_scientific_goal":
        "sakai224_full_vs_Sakai",

    "is_Sakai_final_classifier_replication":
        False,


    # --------------------------------------------------------
    # INPUT
    # --------------------------------------------------------

    "input": {

        "artifact":
            "sakai224_full_mu_epoch100_v1",

        "canonical_feature":
            "mu",

        "latent_dim":
            512,

        "train_rows":
            6866,

        "validation_rows":
            1717,

        "train_sha256":
            EXPECTED_TRAIN_MU_SHA256,

        "validation_sha256":
            EXPECTED_VALIDATION_MU_SHA256,

        "dtype":
            "float32",

        "matrix_layout":
            "row_major",
    },


    # --------------------------------------------------------
    # PRIMARY FEATURE SPACE
    # --------------------------------------------------------

    "primary_feature_space": {

        "name":
            "raw_mu",

        "pca":
            False,

        "standardization":
            False,

        "role":
            (
                "primary diagnostic of the native "
                "encoder geometry"
            ),
    },


    # --------------------------------------------------------
    # PRE-REGISTERED SENSITIVITY ANALYSIS
    # --------------------------------------------------------

    "sensitivity_feature_space": {

        "name":
            "zscore_train_only",

        "enabled":
            True,

        "role":
            "pre_registered_sensitivity_analysis",

        "fit_on":
            "train_only",

        "variance_definition":
            "population_ddof0",

        "validation_refit":
            False,

        "pca":
            False,

        "selection_based_on_labels":
            False,
    },


    # --------------------------------------------------------
    # PRE-REGISTERED k VALUES
    # --------------------------------------------------------

    "k_values": [
        22,
        36,
    ],


    "k_interpretation": {

        "22":
            "Gravity Spy class-count control",

        "36":
            (
                "same nominal class count as Sakai "
                "primary IIC C=36; not equivalent to IIC"
            ),
    },


    # --------------------------------------------------------
    # PYTHON KMEANS
    # --------------------------------------------------------

    "kmeans": {

        "backend":
            "sklearn.cluster.KMeans",

        "sklearn_version":
            sklearn.__version__,

        "init":
            "k-means++",

        "n_init":
            10,

        "random_state":
            123,

        "random_state_role":
            "PROJECT reproducibility only",

        "max_iter":
            300,

        "tol":
            1.0e-4,

        "algorithm":
            "lloyd",

        "copy_x":
            True,

        "verbose":
            0,

        "numeric_thread_limit":
            1,
    },


    # --------------------------------------------------------
    # FIT / VALIDATION POLICY
    # --------------------------------------------------------

    "fit_policy": {

        "fit_split":
            "train",

        "validation_policy":
            "nearest_frozen_train_centroid",

        "validation_refit":
            False,

        "labels_used_for_fit":
            False,

        "labels_used_for_init_selection":
            False,
    },


    # --------------------------------------------------------
    # EVALUATION
    # --------------------------------------------------------

    "evaluation_policy": {

        "internal_metrics": [
            "inertia",
            "silhouette",
            "davies_bouldin",
        ],

        "external_metrics": [
            "purity",
            "nmi_arithmetic",
            "ari",
            "homogeneity",
            "completeness",
        ],

        "labels_used_only_after_assignments_frozen":
            True,

        "internal_and_external_metrics_kept_separate":
            True,
    },


    # --------------------------------------------------------
    # RELATION TO SAKAI
    # --------------------------------------------------------

    "Sakai_relation": {

        "VAE_latent_dim":
            512,

        "IIC_primary_class_count":
            36,

        "kmeans_is_paper_final_method":
            False,

        "primary_future_method":
            "IIC_plus_multi_classifier_plus_spectral_clustering",

        "direct_comparison_to_Sakai_final_accuracy_allowed":
            False,
    },


    # --------------------------------------------------------
    # SEPARATE SECONDARY CROSS-VAE EXPERIMENT
    # --------------------------------------------------------

    "cross_vae_comparison": {

        "role":
            "separate_secondary_experiment",

        "does_not_determine_python_full_protocol":
            True,

        "future_common_backend":
            "C baseline_kmeans",

        "profiles": [
            "mini75",
            "mini112",
            "mini224_1000",
            "sakai224_full",
        ],

        "same_backend_for_all_profiles":
            True,
    },
}


# ============================================================
# CANONICAL CONFIG HASH
# ============================================================

KMEANS_CONFIG_JSON = json.dumps(
    KMEANS_CONFIG,
    sort_keys=True,
    separators=(",", ":"),
    ensure_ascii=True,
)


KMEANS_CONFIG_SHA256 = (
    hashlib.sha256(
        KMEANS_CONFIG_JSON.encode(
            "utf-8"
        )
    ).hexdigest()
)


# ============================================================
# HARD CONTRACT GATES
# ============================================================

goal_gate = (
    KMEANS_CONFIG[
        "main_scientific_goal"
    ]
    == "sakai224_full_vs_Sakai"

    and KMEANS_CONFIG[
        "is_Sakai_final_classifier_replication"
    ]
    is False
)


input_gate = (
    KMEANS_CONFIG[
        "input"
    ][
        "latent_dim"
    ] == 512

    and KMEANS_CONFIG[
        "input"
    ][
        "train_sha256"
    ]
    == EXPECTED_TRAIN_MU_SHA256

    and KMEANS_CONFIG[
        "input"
    ][
        "validation_sha256"
    ]
    == EXPECTED_VALIDATION_MU_SHA256
)


primary_gate = (
    KMEANS_CONFIG[
        "primary_feature_space"
    ][
        "name"
    ] == "raw_mu"

    and KMEANS_CONFIG[
        "primary_feature_space"
    ][
        "pca"
    ] is False

    and KMEANS_CONFIG[
        "primary_feature_space"
    ][
        "standardization"
    ] is False
)


sensitivity_gate = (
    KMEANS_CONFIG[
        "sensitivity_feature_space"
    ][
        "enabled"
    ] is True

    and KMEANS_CONFIG[
        "sensitivity_feature_space"
    ][
        "fit_on"
    ] == "train_only"

    and KMEANS_CONFIG[
        "sensitivity_feature_space"
    ][
        "validation_refit"
    ] is False

    and KMEANS_CONFIG[
        "sensitivity_feature_space"
    ][
        "selection_based_on_labels"
    ] is False
)


k_gate = (
    KMEANS_CONFIG[
        "k_values"
    ] == [
        22,
        36,
    ]
)


algorithm_gate = (
    KMEANS_CONFIG[
        "kmeans"
    ][
        "init"
    ] == "k-means++"

    and KMEANS_CONFIG[
        "kmeans"
    ][
        "n_init"
    ] == 10

    and KMEANS_CONFIG[
        "kmeans"
    ][
        "random_state"
    ] == 123

    and KMEANS_CONFIG[
        "kmeans"
    ][
        "max_iter"
    ] == 300

    and KMEANS_CONFIG[
        "kmeans"
    ][
        "tol"
    ] == 1.0e-4

    and KMEANS_CONFIG[
        "kmeans"
    ][
        "algorithm"
    ] == "lloyd"
)


fit_policy_gate = (
    KMEANS_CONFIG[
        "fit_policy"
    ][
        "fit_split"
    ] == "train"

    and KMEANS_CONFIG[
        "fit_policy"
    ][
        "validation_refit"
    ] is False

    and KMEANS_CONFIG[
        "fit_policy"
    ][
        "labels_used_for_fit"
    ] is False

    and KMEANS_CONFIG[
        "fit_policy"
    ][
        "labels_used_for_init_selection"
    ] is False
)


Sakai_separation_gate = (
    KMEANS_CONFIG[
        "Sakai_relation"
    ][
        "kmeans_is_paper_final_method"
    ] is False

    and KMEANS_CONFIG[
        "Sakai_relation"
    ][
        "direct_comparison_to_Sakai_final_accuracy_allowed"
    ] is False
)


cross_vae_separation_gate = (
    KMEANS_CONFIG[
        "cross_vae_comparison"
    ][
        "role"
    ]
    == "separate_secondary_experiment"

    and KMEANS_CONFIG[
        "cross_vae_comparison"
    ][
        "does_not_determine_python_full_protocol"
    ] is True
)


assert goal_gate
assert input_gate
assert primary_gate
assert sensitivity_gate
assert k_gate
assert algorithm_gate
assert fit_policy_gate
assert Sakai_separation_gate
assert cross_vae_separation_gate


# ============================================================
# REPORT
# ============================================================

print()
print(
    "=== NOTEBOOK 05 CORRECTED FULL-VAE KMEANS CONTRACT ==="
)

print()

print(
    "MAIN_SCIENTIFIC_GOAL =",
    KMEANS_CONFIG[
        "main_scientific_goal"
    ],
)

print(
    "KMEANS_IS_SAKAI_FINAL_METHOD =",
    KMEANS_CONFIG[
        "Sakai_relation"
    ][
        "kmeans_is_paper_final_method"
    ],
)

print()

print(
    "INPUT_ARTIFACT =",
    KMEANS_CONFIG[
        "input"
    ][
        "artifact"
    ],
)

print(
    "PRIMARY_FEATURE_SPACE =",
    KMEANS_CONFIG[
        "primary_feature_space"
    ][
        "name"
    ],
)

print(
    "SENSITIVITY_FEATURE_SPACE =",
    KMEANS_CONFIG[
        "sensitivity_feature_space"
    ][
        "name"
    ],
)

print()

print(
    "K_VALUES =",
    KMEANS_CONFIG[
        "k_values"
    ],
)

print(
    "INIT =",
    KMEANS_CONFIG[
        "kmeans"
    ][
        "init"
    ],
)

print(
    "N_INIT =",
    KMEANS_CONFIG[
        "kmeans"
    ][
        "n_init"
    ],
)

print(
    "RANDOM_STATE =",
    KMEANS_CONFIG[
        "kmeans"
    ][
        "random_state"
    ],
)

print(
    "MAX_ITER =",
    KMEANS_CONFIG[
        "kmeans"
    ][
        "max_iter"
    ],
)

print(
    "TOL =",
    KMEANS_CONFIG[
        "kmeans"
    ][
        "tol"
    ],
)

print(
    "ALGORITHM =",
    KMEANS_CONFIG[
        "kmeans"
    ][
        "algorithm"
    ],
)

print()

print(
    "SUPERSEDED_CONFIG_SHA256 =",
    SUPERSEDED_KMEANS_CONFIG_SHA256,
)

print(
    "CORRECTED_KMEANS_CONFIG_SHA256 =",
    KMEANS_CONFIG_SHA256,
)

print()

print(
    "GOAL_GATE =",
    goal_gate,
)

print(
    "INPUT_GATE =",
    input_gate,
)

print(
    "PRIMARY_GATE =",
    primary_gate,
)

print(
    "SENSITIVITY_GATE =",
    sensitivity_gate,
)

print(
    "K_GATE =",
    k_gate,
)

print(
    "ALGORITHM_GATE =",
    algorithm_gate,
)

print(
    "FIT_POLICY_GATE =",
    fit_policy_gate,
)

print(
    "SAKAI_SEPARATION_GATE =",
    Sakai_separation_gate,
)

print(
    "CROSS_VAE_SEPARATION_GATE =",
    cross_vae_separation_gate,
)

print()

print(
    "LABELS_READ =",
    False,
)

print(
    "KMEANS_EXECUTED =",
    False,
)

print(
    "CLUSTER_ASSIGNMENTS_CREATED =",
    False,
)

print(
    "OFFICIAL_DRIVE_ARTIFACTS_MODIFIED =",
    False,
)

print()

print(
    "NOTEBOOK_05_A1_CORRECTED_CONTRACT_FROZEN =",
    (
        goal_gate
        and input_gate
        and primary_gate
        and sensitivity_gate
        and k_gate
        and algorithm_gate
        and fit_policy_gate
        and Sakai_separation_gate
        and cross_vae_separation_gate
    ),
)


=== NOTEBOOK 05 CORRECTED FULL-VAE KMEANS CONTRACT ===

MAIN_SCIENTIFIC_GOAL = sakai224_full_vs_Sakai
KMEANS_IS_SAKAI_FINAL_METHOD = False

INPUT_ARTIFACT = sakai224_full_mu_epoch100_v1
PRIMARY_FEATURE_SPACE = raw_mu
SENSITIVITY_FEATURE_SPACE = zscore_train_only

K_VALUES = [22, 36]
INIT = k-means++
N_INIT = 10
RANDOM_STATE = 123
MAX_ITER = 300
TOL = 0.0001
ALGORITHM = lloyd

SUPERSEDED_CONFIG_SHA256 = f435f43f7b3afdb813d7b7e9b3c15c2cf84741bba7619ab7d5c8d6cd8349e2ec
CORRECTED_KMEANS_CONFIG_SHA256 = f43a89b318bf22f03a63d2a7e5a995917e97f160c7d8eccbb19fade9a6457d29

GOAL_GATE = True
INPUT_GATE = True
PRIMARY_GATE = True
SENSITIVITY_GATE = True
K_GATE = True
ALGORITHM_GATE = True
FIT_POLICY_GATE = True
SAKAI_SEPARATION_GATE = True
CROSS_VAE_SEPARATION_GATE = True

LABELS_READ = False
KMEANS_EXECUTED = False
CLUSTER_ASSIGNMENTS_CREATED = False
OFFICIAL_DRIVE_ARTIFACTS_MODIFIED = False

NOTEBOOK_05_A1_CORRECTED_CONTRACT_FROZEN = True


## STEP 05-A2 — Fit k-means++ primario sugli embedding raw-$\mu$

L'analisi PROJECT primaria opera sugli embedding canonici del VAE
`sakai224_full`.

Lo spazio delle feature non viene modificato:

$$
X_{\mathrm{train}}
=
M_\mu^{\mathrm{train}}
\in
\mathbb{R}^{6866\times512},
$$

$$
X_{\mathrm{validation}}
=
M_\mu^{\mathrm{validation}}
\in
\mathbb{R}^{1717\times512}.
$$

Non vengono applicate:

$$
\mathrm{PCA},
$$

né

$$
\mathrm{standardizzazione}.
$$

Per ciascun valore

$$
k\in\{22,36\},
$$

il k-means++ viene fit esclusivamente sul training split.

L'obiettivo è:

$$
J
=
\sum_{i=1}^{N_{\mathrm{train}}}
\min_{j\in\{1,\ldots,k\}}
\left\|
x_i-c_j
\right\|_2^2.
$$

Per ridurre la dipendenza da un singolo minimo locale vengono effettuate

$$
n_{\mathrm{init}}=10
$$

inizializzazioni k-means++ e viene mantenuta automaticamente la soluzione
con inerzia minima.

I centroidi finali vengono quindi congelati.

Gli eventi della validation vengono assegnati al centroide train più vicino:

$$
a_i^{\mathrm{validation}}
=
\operatorname*{argmin}_{j}
\left\|
x_i^{\mathrm{validation}}-c_j
\right\|_2^2.
$$

La validation non partecipa al fit e non viene effettuato alcun refit.

Il fit verifica:

- convergenza entro il limite di 300 iterazioni;
- finitezza dei centroidi;
- assenza di cluster train vuoti;
- coerenza dell'inertia;
- corretta assegnazione della validation ai centroidi train congelati;
- immutabilità degli embedding sorgente;
- riproducibilità della stessa configurazione.

Le label Gravity Spy non vengono lette.

Il risultato del k-means non viene confrontato direttamente con l'accuracy
finale di Sakai, perché il k-means non è il classificatore finale del paper.

In [4]:
# ============================================================
# NOTEBOOK 05 — PRIMARY RAW-MU KMEANS
# STEP 05-A2
#
# k = 22 and k = 36
# n_init = 10
#
# FIT TRAIN ONLY.
# VALIDATION -> frozen train centroids.
#
# NO LABELS.
# NO PERSISTENT WRITES.
# ============================================================

import hashlib
import time

import numpy as np

from sklearn.cluster import KMeans
from threadpoolctl import threadpool_limits


# ============================================================
# PRIVATE INPUT COPIES
#
# Published embedding files remain read-only.
# ============================================================

X_train_raw = np.array(
    train_mu_published,
    dtype=np.float32,
    order="C",
    copy=True,
)

X_validation_raw = np.array(
    validation_mu_published,
    dtype=np.float32,
    order="C",
    copy=True,
)


assert X_train_raw.shape == (
    6866,
    512,
)

assert X_validation_raw.shape == (
    1717,
    512,
)

assert X_train_raw.dtype == np.float32
assert X_validation_raw.dtype == np.float32


# ============================================================
# INPUT BYTE IDENTITIES
# ============================================================

train_input_sha_before = hashlib.sha256(
    X_train_raw.tobytes(
        order="C"
    )
).hexdigest()


validation_input_sha_before = hashlib.sha256(
    X_validation_raw.tobytes(
        order="C"
    )
).hexdigest()


assert (
    train_input_sha_before
    == EXPECTED_TRAIN_MU_SHA256
)

assert (
    validation_input_sha_before
    == EXPECTED_VALIDATION_MU_SHA256
)


# ============================================================
# FROZEN ESTIMATOR FACTORY
# ============================================================

def make_primary_kmeans(k):

    return KMeans(
        n_clusters=k,
        init="k-means++",
        n_init=10,
        max_iter=300,
        tol=1.0e-4,
        verbose=0,
        random_state=123,
        copy_x=True,
        algorithm="lloyd",
    )


# ============================================================
# INDEPENDENT ASSIGNMENT TO FROZEN CENTROIDS
#
# Squared Euclidean distance:
#
# ||x-c||^2 = ||x||^2 + ||c||^2 - 2 x.c
#
# float64 is used only for this audit calculation.
# ============================================================

def assign_to_frozen_centroids(
    X,
    centroids,
    batch_size=512,
):

    X = np.asarray(
        X,
        dtype=np.float32,
    )

    centroids64 = np.asarray(
        centroids,
        dtype=np.float64,
    )


    n_samples = X.shape[0]


    assignments = np.empty(
        n_samples,
        dtype=np.int32,
    )


    squared_distances = np.empty(
        n_samples,
        dtype=np.float64,
    )


    centroid_norm2 = np.sum(
        centroids64 * centroids64,
        axis=1,
    )


    for start in range(
        0,
        n_samples,
        batch_size,
    ):

        stop = min(
            start + batch_size,
            n_samples,
        )


        X_batch = np.asarray(
            X[start:stop],
            dtype=np.float64,
        )


        x_norm2 = np.sum(
            X_batch * X_batch,
            axis=1,
            keepdims=True,
        )


        distances2 = (
            x_norm2
            + centroid_norm2[None, :]
            - 2.0
            * (
                X_batch
                @ centroids64.T
            )
        )


        # Remove tiny negative values caused only by
        # floating-point roundoff.
        np.maximum(
            distances2,
            0.0,
            out=distances2,
        )


        batch_assignment = np.argmin(
            distances2,
            axis=1,
        )


        assignments[
            start:stop
        ] = batch_assignment.astype(
            np.int32,
            copy=False,
        )


        squared_distances[
            start:stop
        ] = distances2[
            np.arange(
                stop - start
            ),
            batch_assignment,
        ]


    return (
        assignments,
        squared_distances,
    )


# ============================================================
# COMPLETE FIT + VALIDATION ASSIGNMENT
# ============================================================

def run_primary_raw_mu_kmeans(k):

    estimator = make_primary_kmeans(
        k
    )


    start_time = time.perf_counter()


    # Single numerical thread:
    # reduces avoidable environment-dependent variation.
    with threadpool_limits(
        limits=1
    ):

        estimator.fit(
            X_train_raw
        )


    elapsed_seconds = (
        time.perf_counter()
        - start_time
    )


    # ========================================================
    # FINAL TRAIN CENTROIDS
    # ========================================================

    centroids = np.asarray(
        estimator.cluster_centers_,
        dtype=np.float32,
        order="C",
    )


    sklearn_train_assignments = np.asarray(
        estimator.labels_,
        dtype=np.int32,
    )


    # ========================================================
    # INDEPENDENT TRAIN / VALIDATION ASSIGNMENT
    # ========================================================

    (
        train_assignments,
        train_squared_distances,
    ) = assign_to_frozen_centroids(
        X_train_raw,
        centroids,
    )


    (
        validation_assignments,
        validation_squared_distances,
    ) = assign_to_frozen_centroids(
        X_validation_raw,
        centroids,
    )


    # ========================================================
    # TRAIN LABEL CONSISTENCY
    # ========================================================

    train_assignment_gate = np.array_equal(
        train_assignments,
        sklearn_train_assignments,
    )


    assert train_assignment_gate


    # ========================================================
    # INERTIA AUDIT
    # ========================================================

    sklearn_train_inertia = float(
        estimator.inertia_
    )


    recomputed_train_inertia = float(
        np.sum(
            train_squared_distances,
            dtype=np.float64,
        )
    )


    validation_inertia = float(
        np.sum(
            validation_squared_distances,
            dtype=np.float64,
        )
    )


    inertia_relative_error = (
        abs(
            recomputed_train_inertia
            - sklearn_train_inertia
        )
        /
        max(
            abs(
                recomputed_train_inertia
            ),
            1.0,
        )
    )


    # sklearn fit uses float32 input while the independent
    # audit recomputes squared distances in float64.
    inertia_gate = (
        inertia_relative_error
        < 1.0e-5
    )


    assert inertia_gate


    # ========================================================
    # CLUSTER SIZES
    # ========================================================

    train_cluster_sizes = np.bincount(
        train_assignments,
        minlength=k,
    ).astype(
        np.int64,
    )


    validation_cluster_sizes = np.bincount(
        validation_assignments,
        minlength=k,
    ).astype(
        np.int64,
    )


    no_empty_train_cluster_gate = bool(
        np.all(
            train_cluster_sizes > 0
        )
    )


    assert no_empty_train_cluster_gate


    validation_count_gate = (
        int(
            validation_cluster_sizes.sum()
        )
        == 1717
    )


    assert validation_count_gate


    # ========================================================
    # NUMERIC / CONVERGENCE GATES
    # ========================================================

    numeric_gate = (
        np.isfinite(
            centroids
        ).all()

        and np.isfinite(
            train_squared_distances
        ).all()

        and np.isfinite(
            validation_squared_distances
        ).all()

        and np.isfinite(
            sklearn_train_inertia
        )

        and np.isfinite(
            recomputed_train_inertia
        )

        and np.isfinite(
            validation_inertia
        )
    )


    convergence_gate = (
        1
        <= int(
            estimator.n_iter_
        )
        <= 300
    )


    assert numeric_gate
    assert convergence_gate


    # ========================================================
    # BYTE IDENTITIES
    # ========================================================

    centroid_sha256 = hashlib.sha256(
        centroids.astype(
            "<f4",
            copy=False,
        ).tobytes(
            order="C"
        )
    ).hexdigest()


    train_assignment_sha256 = hashlib.sha256(
        train_assignments.astype(
            "<i4",
            copy=False,
        ).tobytes(
            order="C"
        )
    ).hexdigest()


    validation_assignment_sha256 = hashlib.sha256(
        validation_assignments.astype(
            "<i4",
            copy=False,
        ).tobytes(
            order="C"
        )
    ).hexdigest()


    return {

        "k":
            k,

        "estimator":
            estimator,

        "centroids":
            centroids,

        "train_assignments":
            train_assignments,

        "validation_assignments":
            validation_assignments,

        "train_squared_distances":
            train_squared_distances,

        "validation_squared_distances":
            validation_squared_distances,

        "train_cluster_sizes":
            train_cluster_sizes,

        "validation_cluster_sizes":
            validation_cluster_sizes,

        "iterations":
            int(
                estimator.n_iter_
            ),

        "elapsed_seconds":
            float(
                elapsed_seconds
            ),

        "sklearn_train_inertia":
            sklearn_train_inertia,

        "train_inertia":
            recomputed_train_inertia,

        "validation_inertia":
            validation_inertia,

        "inertia_relative_error":
            float(
                inertia_relative_error
            ),

        "centroid_sha256":
            centroid_sha256,

        "train_assignment_sha256":
            train_assignment_sha256,

        "validation_assignment_sha256":
            validation_assignment_sha256,

        "train_assignment_gate":
            train_assignment_gate,

        "inertia_gate":
            inertia_gate,

        "no_empty_train_cluster_gate":
            no_empty_train_cluster_gate,

        "validation_count_gate":
            validation_count_gate,

        "numeric_gate":
            numeric_gate,

        "convergence_gate":
            convergence_gate,
    }


# ============================================================
# PRIMARY FITS
# ============================================================

print(
    "Running primary raw_mu k=22..."
)

raw_mu_k22 = run_primary_raw_mu_kmeans(
    22
)


print(
    "Running primary raw_mu k=36..."
)

raw_mu_k36 = run_primary_raw_mu_kmeans(
    36
)


# ============================================================
# FULL REPRODUCIBILITY REFITS
#
# Same bytes + same explicit config + same thread limit.
# ============================================================

print(
    "Repeating raw_mu k=22..."
)

raw_mu_k22_repeat = (
    run_primary_raw_mu_kmeans(
        22
    )
)


print(
    "Repeating raw_mu k=36..."
)

raw_mu_k36_repeat = (
    run_primary_raw_mu_kmeans(
        36
    )
)


# ============================================================
# REPRODUCIBILITY CONTRACT
# ============================================================

def exact_reproducibility_gate(
    result_a,
    result_b,
):

    return (
        result_a[
            "centroid_sha256"
        ]
        == result_b[
            "centroid_sha256"
        ]

        and result_a[
            "train_assignment_sha256"
        ]
        == result_b[
            "train_assignment_sha256"
        ]

        and result_a[
            "validation_assignment_sha256"
        ]
        == result_b[
            "validation_assignment_sha256"
        ]

        and result_a[
            "iterations"
        ]
        == result_b[
            "iterations"
        ]

        and result_a[
            "train_inertia"
        ]
        == result_b[
            "train_inertia"
        ]

        and result_a[
            "validation_inertia"
        ]
        == result_b[
            "validation_inertia"
        ]
    )


k22_reproducibility_gate = (
    exact_reproducibility_gate(
        raw_mu_k22,
        raw_mu_k22_repeat,
    )
)


k36_reproducibility_gate = (
    exact_reproducibility_gate(
        raw_mu_k36,
        raw_mu_k36_repeat,
    )
)


assert k22_reproducibility_gate
assert k36_reproducibility_gate


# ============================================================
# INPUT IMMUTABILITY
# ============================================================

train_input_sha_after = hashlib.sha256(
    X_train_raw.tobytes(
        order="C"
    )
).hexdigest()


validation_input_sha_after = hashlib.sha256(
    X_validation_raw.tobytes(
        order="C"
    )
).hexdigest()


input_immutability_gate = (
    train_input_sha_before
    == train_input_sha_after
    == EXPECTED_TRAIN_MU_SHA256

    and validation_input_sha_before
    == validation_input_sha_after
    == EXPECTED_VALIDATION_MU_SHA256
)


assert input_immutability_gate


# ============================================================
# REPORT
# ============================================================

def report_run(
    title,
    result,
):

    print()
    print(title)

    print(
        "  k =",
        result["k"],
    )

    print(
        "  n_init = 10"
    )

    print(
        "  iterations_of_selected_solution =",
        result["iterations"],
    )

    print(
        "  elapsed_seconds =",
        result["elapsed_seconds"],
    )

    print(
        "  sklearn_train_inertia =",
        result[
            "sklearn_train_inertia"
        ],
    )

    print(
        "  recomputed_train_inertia =",
        result[
            "train_inertia"
        ],
    )

    print(
        "  validation_inertia =",
        result[
            "validation_inertia"
        ],
    )

    print(
        "  inertia_relative_error =",
        result[
            "inertia_relative_error"
        ],
    )

    print(
        "  min_train_cluster =",
        int(
            result[
                "train_cluster_sizes"
            ].min()
        ),
    )

    print(
        "  max_train_cluster =",
        int(
            result[
                "train_cluster_sizes"
            ].max()
        ),
    )

    print(
        "  empty_validation_clusters =",
        int(
            np.sum(
                result[
                    "validation_cluster_sizes"
                ] == 0
            )
        ),
    )

    print(
        "  centroid_sha256 =",
        result[
            "centroid_sha256"
        ],
    )

    print(
        "  train_assignment_sha256 =",
        result[
            "train_assignment_sha256"
        ],
    )

    print(
        "  validation_assignment_sha256 =",
        result[
            "validation_assignment_sha256"
        ],
    )


print()
print(
    "=== NOTEBOOK 05 PRIMARY RAW-MU KMEANS ==="
)


report_run(
    "RAW_MU_K22",
    raw_mu_k22,
)


report_run(
    "RAW_MU_K36",
    raw_mu_k36,
)


print()

print(
    "K22_REPRODUCIBILITY_GATE =",
    k22_reproducibility_gate,
)

print(
    "K36_REPRODUCIBILITY_GATE =",
    k36_reproducibility_gate,
)

print(
    "INPUT_IMMUTABILITY_GATE =",
    input_immutability_gate,
)

print()

print(
    "LABELS_READ =",
    False,
)

print(
    "VALIDATION_REFIT =",
    False,
)

print(
    "PCA_APPLIED =",
    False,
)

print(
    "STANDARDIZATION_APPLIED =",
    False,
)

print(
    "CLUSTERING_PUBLISHED =",
    False,
)

print(
    "OFFICIAL_DRIVE_ARTIFACTS_MODIFIED =",
    False,
)

print()

print(
    "NOTEBOOK_05_A2_RAW_MU_KMEANS_PASS =",
    (
        raw_mu_k22[
            "train_assignment_gate"
        ]

        and raw_mu_k22[
            "inertia_gate"
        ]

        and raw_mu_k22[
            "no_empty_train_cluster_gate"
        ]

        and raw_mu_k22[
            "validation_count_gate"
        ]

        and raw_mu_k22[
            "numeric_gate"
        ]

        and raw_mu_k22[
            "convergence_gate"
        ]

        and raw_mu_k36[
            "train_assignment_gate"
        ]

        and raw_mu_k36[
            "inertia_gate"
        ]

        and raw_mu_k36[
            "no_empty_train_cluster_gate"
        ]

        and raw_mu_k36[
            "validation_count_gate"
        ]

        and raw_mu_k36[
            "numeric_gate"
        ]

        and raw_mu_k36[
            "convergence_gate"
        ]

        and k22_reproducibility_gate
        and k36_reproducibility_gate
        and input_immutability_gate
    ),
)

Running primary raw_mu k=22...
Running primary raw_mu k=36...
Repeating raw_mu k=22...
Repeating raw_mu k=36...

=== NOTEBOOK 05 PRIMARY RAW-MU KMEANS ===

RAW_MU_K22
  k = 22
  n_init = 10
  iterations_of_selected_solution = 23
  elapsed_seconds = 3.245002695999574
  sklearn_train_inertia = 352788.5625
  recomputed_train_inertia = 352788.44498125097
  validation_inertia = 88664.65157719958
  inertia_relative_error = 3.3311394038454644e-07
  min_train_cluster = 31
  max_train_cluster = 1105
  empty_validation_clusters = 0
  centroid_sha256 = 7c196346d32e08d12ebdae94ed602b588827f43b98479eb2dd26fa92ced776f9
  train_assignment_sha256 = 13e7f4ed87d3ed459f9e6b7fdf706e1b3758bac735444131e99ad69c92791cf4
  validation_assignment_sha256 = f3ca101ee4078814378dee3874d8feb04d89f58c89af95088fb4c3801e39db5f

RAW_MU_K36
  k = 36
  n_init = 10
  iterations_of_selected_solution = 54
  elapsed_seconds = 4.504332259999501
  sklearn_train_inertia = 290812.09375
  recomputed_train_inertia = 290812.185252772

## STEP 05-A3 — Diagnostiche interne label-free del clustering raw-$\mu$

Prima di introdurre le label Gravity Spy, vengono valutate alcune proprietà
geometriche delle partizioni ottenute nello STEP 05-A2.

Le diagnostiche sono interamente unsupervised.

Per ogni configurazione

$$
k\in\{22,36\},
$$

vengono calcolate separatamente sul training e sulla validation:

1. inertia media per evento;
2. silhouette coefficient;
3. Davies–Bouldin index;
4. distribuzione delle dimensioni dei cluster.

### Silhouette coefficient

Per ogni evento $i$:

$$
s(i)
=
\frac{b(i)-a(i)}
{\max\{a(i),b(i)\}},
$$

dove:

- $a(i)$ è la distanza media dagli eventi appartenenti allo stesso cluster;
- $b(i)$ è la minima distanza media verso un altro cluster.

Il valore medio soddisfa:

$$
-1\leq S\leq1.
$$

Valori maggiori indicano, in termini puramente geometrici, gruppi più
compatti e separati.

La silhouette viene calcolata sull'intero split, senza sottocampionamento.

### Davies–Bouldin index

Il Davies–Bouldin index confronta dispersione intra-cluster e separazione
fra cluster.

Valori più bassi indicano una geometria più compatta e separata.

Il valore viene calcolato con la definizione standard di scikit-learn sulla
partizione osservata nello split considerato.

### Interpretazione

Queste diagnostiche non utilizzano le classi Gravity Spy e non misurano
corrispondenza semantica con le morfologie note.

In particolare:

$$
\text{buona geometria interna}
\not\Rightarrow
\text{buona corrispondenza con le 22 label}.
$$

Allo stesso modo, il confronto fra $k=22$ e $k=36$ non verrà deciso usando
un singolo indicatore.

Le label resteranno non lette anche dopo questo step.

In [5]:
# ============================================================
# NOTEBOOK 05 — RAW-MU INTERNAL CLUSTER DIAGNOSTICS
# STEP 05-A3
#
# LABEL-FREE.
# NO Drive writes.
# ============================================================

import time

import numpy as np

from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
)

from threadpoolctl import threadpool_limits


# ============================================================
# INTERNAL DIAGNOSTIC FUNCTION
# ============================================================

def compute_internal_diagnostics(
    X,
    assignments,
    inertia,
    *,
    name,
):

    X = np.asarray(
        X,
        dtype=np.float32,
        order="C",
    )

    assignments = np.asarray(
        assignments,
        dtype=np.int32,
    )


    assert X.shape[0] == assignments.shape[0]

    unique_clusters = np.unique(
        assignments
    )

    assert unique_clusters.size >= 2


    # --------------------------------------------------------
    # INERTIA PER EVENT
    # --------------------------------------------------------

    inertia_per_event = (
        float(inertia)
        / float(X.shape[0])
    )


    # --------------------------------------------------------
    # SILHOUETTE — FULL SPLIT
    # --------------------------------------------------------

    silhouette_start = time.perf_counter()


    with threadpool_limits(
        limits=1
    ):

        silhouette = float(
            silhouette_score(
                X,
                assignments,
                metric="euclidean",
            )
        )


    silhouette_seconds = (
        time.perf_counter()
        - silhouette_start
    )


    # --------------------------------------------------------
    # DAVIES–BOULDIN
    # --------------------------------------------------------

    db_start = time.perf_counter()


    with threadpool_limits(
        limits=1
    ):

        davies_bouldin = float(
            davies_bouldin_score(
                X,
                assignments,
            )
        )


    db_seconds = (
        time.perf_counter()
        - db_start
    )


    # --------------------------------------------------------
    # OCCUPANCY
    # --------------------------------------------------------

    k = int(
        assignments.max()
        + 1
    )


    cluster_sizes = np.bincount(
        assignments,
        minlength=k,
    ).astype(
        np.int64,
    )


    cluster_fractions = (
        cluster_sizes
        / float(
            X.shape[0]
        )
    )


    occupancy_quantiles = np.quantile(
        cluster_sizes,
        [
            0.0,
            0.25,
            0.50,
            0.75,
            1.0,
        ],
    )


    # Shannon effective number of occupied clusters:
    #
    # exp(-sum p_j log p_j)
    #
    # Descriptive only, not a pre-registered selection metric.
    positive_fractions = cluster_fractions[
        cluster_fractions > 0.0
    ]


    occupancy_entropy = float(
        -np.sum(
            positive_fractions
            * np.log(
                positive_fractions
            )
        )
    )


    effective_cluster_count = float(
        np.exp(
            occupancy_entropy
        )
    )


    numeric_gate = (
        np.isfinite(
            inertia_per_event
        )

        and np.isfinite(
            silhouette
        )

        and np.isfinite(
            davies_bouldin
        )

        and -1.0
        <= silhouette
        <= 1.0

        and davies_bouldin
        >= 0.0

        and int(
            cluster_sizes.sum()
        )
        == X.shape[0]
    )


    assert numeric_gate


    return {

        "name":
            name,

        "n_samples":
            int(
                X.shape[0]
            ),

        "n_features":
            int(
                X.shape[1]
            ),

        "n_clusters":
            int(
                unique_clusters.size
            ),

        "inertia":
            float(
                inertia
            ),

        "inertia_per_event":
            inertia_per_event,

        "silhouette":
            silhouette,

        "silhouette_seconds":
            float(
                silhouette_seconds
            ),

        "davies_bouldin":
            davies_bouldin,

        "davies_bouldin_seconds":
            float(
                db_seconds
            ),

        "cluster_sizes":
            cluster_sizes,

        "cluster_fractions":
            cluster_fractions,

        "occupancy_quantiles":
            occupancy_quantiles,

        "occupancy_entropy":
            occupancy_entropy,

        "effective_cluster_count":
            effective_cluster_count,

        "numeric_gate":
            numeric_gate,
    }


# ============================================================
# RUN ALL FOUR LABEL-FREE DIAGNOSTICS
# ============================================================

print(
    "Computing raw_mu k=22 train diagnostics..."
)

raw_mu_k22_train_internal = (
    compute_internal_diagnostics(
        X_train_raw,
        raw_mu_k22[
            "train_assignments"
        ],
        raw_mu_k22[
            "train_inertia"
        ],
        name="raw_mu_k22_train",
    )
)


print(
    "Computing raw_mu k=22 validation diagnostics..."
)

raw_mu_k22_validation_internal = (
    compute_internal_diagnostics(
        X_validation_raw,
        raw_mu_k22[
            "validation_assignments"
        ],
        raw_mu_k22[
            "validation_inertia"
        ],
        name="raw_mu_k22_validation",
    )
)


print(
    "Computing raw_mu k=36 train diagnostics..."
)

raw_mu_k36_train_internal = (
    compute_internal_diagnostics(
        X_train_raw,
        raw_mu_k36[
            "train_assignments"
        ],
        raw_mu_k36[
            "train_inertia"
        ],
        name="raw_mu_k36_train",
    )
)


print(
    "Computing raw_mu k=36 validation diagnostics..."
)

raw_mu_k36_validation_internal = (
    compute_internal_diagnostics(
        X_validation_raw,
        raw_mu_k36[
            "validation_assignments"
        ],
        raw_mu_k36[
            "validation_inertia"
        ],
        name="raw_mu_k36_validation",
    )
)


# ============================================================
# REPORT
# ============================================================

def report_internal(
    result,
):

    print()
    print(
        result["name"]
    )

    print(
        "  n_samples =",
        result["n_samples"],
    )

    print(
        "  n_clusters =",
        result["n_clusters"],
    )

    print(
        "  inertia_per_event =",
        result[
            "inertia_per_event"
        ],
    )

    print(
        "  silhouette =",
        result[
            "silhouette"
        ],
    )

    print(
        "  silhouette_seconds =",
        result[
            "silhouette_seconds"
        ],
    )

    print(
        "  davies_bouldin =",
        result[
            "davies_bouldin"
        ],
    )

    print(
        "  cluster_size_quantiles[min,q25,median,q75,max] =",
        result[
            "occupancy_quantiles"
        ].tolist(),
    )

    print(
        "  effective_cluster_count =",
        result[
            "effective_cluster_count"
        ],
    )


print()
print(
    "=== NOTEBOOK 05 RAW-MU INTERNAL DIAGNOSTICS ==="
)


report_internal(
    raw_mu_k22_train_internal
)

report_internal(
    raw_mu_k22_validation_internal
)

report_internal(
    raw_mu_k36_train_internal
)

report_internal(
    raw_mu_k36_validation_internal
)


# ============================================================
# GENERALIZATION DESCRIPTORS
# ============================================================

k22_inertia_gap_relative = (
    raw_mu_k22_validation_internal[
        "inertia_per_event"
    ]
    /
    raw_mu_k22_train_internal[
        "inertia_per_event"
    ]
    - 1.0
)


k36_inertia_gap_relative = (
    raw_mu_k36_validation_internal[
        "inertia_per_event"
    ]
    /
    raw_mu_k36_train_internal[
        "inertia_per_event"
    ]
    - 1.0
)


print()

print(
    "K22_INERTIA_PER_EVENT_RELATIVE_GAP =",
    k22_inertia_gap_relative,
)

print(
    "K36_INERTIA_PER_EVENT_RELATIVE_GAP =",
    k36_inertia_gap_relative,
)


# ============================================================
# HARD GATE
# ============================================================

internal_gate = all([
    raw_mu_k22_train_internal[
        "numeric_gate"
    ],

    raw_mu_k22_validation_internal[
        "numeric_gate"
    ],

    raw_mu_k36_train_internal[
        "numeric_gate"
    ],

    raw_mu_k36_validation_internal[
        "numeric_gate"
    ],
])


assert internal_gate


print()

print(
    "LABELS_READ =",
    False,
)

print(
    "CLUSTERING_PUBLISHED =",
    False,
)

print(
    "OFFICIAL_DRIVE_ARTIFACTS_MODIFIED =",
    False,
)

print()

print(
    "NOTEBOOK_05_A3_INTERNAL_DIAGNOSTICS_PASS =",
    internal_gate,
)

Computing raw_mu k=22 train diagnostics...
Computing raw_mu k=22 validation diagnostics...
Computing raw_mu k=36 train diagnostics...
Computing raw_mu k=36 validation diagnostics...

=== NOTEBOOK 05 RAW-MU INTERNAL DIAGNOSTICS ===

raw_mu_k22_train
  n_samples = 6866
  n_clusters = 22
  inertia_per_event = 51.38194654547786
  silhouette = 0.1683903932571411
  silhouette_seconds = 1.8012427560006472
  davies_bouldin = 1.7604995849382623
  cluster_size_quantiles[min,q25,median,q75,max] = [31.0, 133.75, 205.0, 373.0, 1105.0]
  effective_cluster_count = 15.590916348961702

raw_mu_k22_validation
  n_samples = 1717
  n_clusters = 22
  inertia_per_event = 51.63928455282445
  silhouette = 0.16647525131702423
  silhouette_seconds = 0.1136191929999768
  davies_bouldin = 1.7538349551203487
  cluster_size_quantiles[min,q25,median,q75,max] = [7.0, 30.25, 50.0, 104.0, 252.0]
  effective_cluster_count = 15.54132958360982

raw_mu_k36_train
  n_samples = 6866
  n_clusters = 36
  inertia_per_event = 42.

### Aggiornamento del protocollo di valutazione esterna

La versione iniziale della roadmap prevedeva la valutazione con le label
Gravity Spy immediatamente dopo il k-means.

Prima di leggere qualsiasi label, il protocollo è stato reso più conservativo.

Le label verranno quindi mantenute fuori dall'analisi fino al congelamento
dell'intera pipeline unsupervised principale:

$$
\mathrm{VAE}
\longrightarrow
\mu
\longrightarrow
\mathrm{IIC}
\longrightarrow
\mathrm{over\text{-}clustering}
\longrightarrow
K=5
\longrightarrow
\mathrm{spectral\ clustering}.
$$

Anche le partizioni k-means verranno congelate prima dell'accesso alle label.

Soltanto nella fase finale saranno costruite le matrici di contingenza e
calcolate le metriche esterne:

$$
\mathrm{Purity},
\qquad
\mathrm{NMI},
\qquad
\mathrm{ARI},
\qquad
\mathrm{Homogeneity},
\qquad
\mathrm{Completeness}.
$$

Questo aggiornamento è stato effettuato quando:

- nessuna label Gravity Spy era stata letta nel Notebook 05;
- nessuna metrica esterna era stata calcolata;
- nessun iperparametro IIC o spectral era stato modificato sulla base delle
  classi note.

Lo scopo è ridurre il rischio di leakage metodologico e di tuning a
posteriori sulle label.

## STEP 05-A4a — Sensitivity analysis: standardizzazione train-only

La seconda geometria delle feature era stata pre-registrata prima di
osservare qualsiasi label.

Per ogni dimensione latente $j$ vengono stimate esclusivamente sul training:

$$
\bar{\mu}^{\mathrm{train}}_j
=
\frac{1}{N_{\mathrm{train}}}
\sum_i
\mu_{i,j},
$$

e la varianza population:

$$
v_j^{\mathrm{train}}
=
\frac{1}{N_{\mathrm{train}}}
\sum_i
\left(
\mu_{i,j}-\bar{\mu}^{\mathrm{train}}_j
\right)^2.
$$

Si definisce:

$$
s_j^{\mathrm{train}}
=
\sqrt{
v_j^{\mathrm{train}}
}.
$$

La trasformazione è quindi:

$$
z_{i,j}
=
\frac{
\mu_{i,j}-\bar{\mu}^{\mathrm{train}}_j
}{
s_j^{\mathrm{train}}
}.
$$

Gli stessi parametri vengono applicati alla validation senza alcun refit.

Questa analisi è una sensitivity analysis PROJECT e non sostituisce lo
spazio canonico raw-$\mu$.

La standardizzazione modifica infatti la metrica euclidea: dimensioni con
varianza originariamente piccola ricevono lo stesso peso marginale delle
dimensioni con varianza elevata.

Per questo i risultati raw-$\mu$ restano l'analisi primaria.

Non viene applicata PCA e non vengono lette label.

In [6]:
# ============================================================
# NOTEBOOK 05 — TRAIN-ONLY Z-SCORE FEATURE SPACE
# STEP 05-A4a
#
# Sensitivity analysis.
#
# FIT scaler on TRAIN only.
# TRANSFORM validation without refit.
#
# NO clustering.
# NO PCA.
# NO labels.
# NO Drive writes.
# ============================================================

import hashlib

import numpy as np

from sklearn.preprocessing import StandardScaler


# ============================================================
# ORIGINAL INPUT IDENTITIES
# ============================================================

train_raw_sha_before_a4 = hashlib.sha256(
    X_train_raw.tobytes(
        order="C"
    )
).hexdigest()

validation_raw_sha_before_a4 = hashlib.sha256(
    X_validation_raw.tobytes(
        order="C"
    )
).hexdigest()


assert (
    train_raw_sha_before_a4
    == EXPECTED_TRAIN_MU_SHA256
)

assert (
    validation_raw_sha_before_a4
    == EXPECTED_VALIDATION_MU_SHA256
)


# ============================================================
# TRAIN-ONLY FIT
#
# sklearn StandardScaler uses population variance (ddof=0).
# ============================================================

zscore_scaler = StandardScaler(
    copy=True,
    with_mean=True,
    with_std=True,
)


zscore_scaler.fit(
    X_train_raw
)


scaler_fit_count_gate = (
    int(
        zscore_scaler.n_samples_seen_
    )
    == 6866
)


assert scaler_fit_count_gate


# Freeze scaler statistics before touching validation.
scaler_mean_before_validation = np.array(
    zscore_scaler.mean_,
    dtype=np.float64,
    copy=True,
)

scaler_var_before_validation = np.array(
    zscore_scaler.var_,
    dtype=np.float64,
    copy=True,
)

scaler_scale_before_validation = np.array(
    zscore_scaler.scale_,
    dtype=np.float64,
    copy=True,
)


# ============================================================
# VARIANCE CONTRACT
# ============================================================

nonconstant_gate = bool(
    np.all(
        scaler_var_before_validation > 0.0
    )
)

finite_scaler_gate = bool(
    np.isfinite(
        scaler_mean_before_validation
    ).all()

    and np.isfinite(
        scaler_var_before_validation
    ).all()

    and np.isfinite(
        scaler_scale_before_validation
    ).all()

    and np.all(
        scaler_scale_before_validation > 0.0
    )
)


assert nonconstant_gate
assert finite_scaler_gate


# ============================================================
# TRANSFORM
# ============================================================

X_train_zscore = np.asarray(
    zscore_scaler.transform(
        X_train_raw
    ),
    dtype=np.float32,
    order="C",
)


X_validation_zscore = np.asarray(
    zscore_scaler.transform(
        X_validation_raw
    ),
    dtype=np.float32,
    order="C",
)


assert X_train_zscore.shape == (
    6866,
    512,
)

assert X_validation_zscore.shape == (
    1717,
    512,
)


# ============================================================
# PROVE VALIDATION DID NOT MODIFY / REFIT THE SCALER
# ============================================================

scaler_immutability_gate = (
    np.array_equal(
        scaler_mean_before_validation,
        zscore_scaler.mean_,
    )

    and np.array_equal(
        scaler_var_before_validation,
        zscore_scaler.var_,
    )

    and np.array_equal(
        scaler_scale_before_validation,
        zscore_scaler.scale_,
    )

    and int(
        zscore_scaler.n_samples_seen_
    )
    == 6866
)


assert scaler_immutability_gate


# ============================================================
# TRAIN STANDARDIZATION AUDIT
#
# Recompute statistics in float64.
# ============================================================

train_zscore_mean = np.mean(
    X_train_zscore,
    axis=0,
    dtype=np.float64,
)


train_zscore_std = np.std(
    X_train_zscore,
    axis=0,
    ddof=0,
    dtype=np.float64,
)


max_abs_train_mean = float(
    np.max(
        np.abs(
            train_zscore_mean
        )
    )
)


max_abs_train_std_error = float(
    np.max(
        np.abs(
            train_zscore_std - 1.0
        )
    )
)


train_standardization_gate = (
    max_abs_train_mean
    < 1.0e-5

    and max_abs_train_std_error
    < 1.0e-5
)


assert train_standardization_gate


# ============================================================
# VALIDATION DESCRIPTIVE STATISTICS
#
# These are NOT forced to mean 0 / std 1.
# ============================================================

validation_zscore_mean = np.mean(
    X_validation_zscore,
    axis=0,
    dtype=np.float64,
)


validation_zscore_std = np.std(
    X_validation_zscore,
    axis=0,
    ddof=0,
    dtype=np.float64,
)


validation_mean_abs_median = float(
    np.median(
        np.abs(
            validation_zscore_mean
        )
    )
)


validation_std_median = float(
    np.median(
        validation_zscore_std
    )
)


# ============================================================
# NUMERIC GATE
# ============================================================

numeric_gate_a4 = (
    np.isfinite(
        X_train_zscore
    ).all()

    and np.isfinite(
        X_validation_zscore
    ).all()

    and np.isfinite(
        validation_zscore_mean
    ).all()

    and np.isfinite(
        validation_zscore_std
    ).all()
)


assert numeric_gate_a4


# ============================================================
# HASH TRANSFORMED FEATURE MATRICES
# ============================================================

train_zscore_sha256 = hashlib.sha256(
    X_train_zscore.astype(
        "<f4",
        copy=False,
    ).tobytes(
        order="C"
    )
).hexdigest()


validation_zscore_sha256 = hashlib.sha256(
    X_validation_zscore.astype(
        "<f4",
        copy=False,
    ).tobytes(
        order="C"
    )
).hexdigest()


# ============================================================
# ORIGINAL INPUT IMMUTABILITY
# ============================================================

train_raw_sha_after_a4 = hashlib.sha256(
    X_train_raw.tobytes(
        order="C"
    )
).hexdigest()


validation_raw_sha_after_a4 = hashlib.sha256(
    X_validation_raw.tobytes(
        order="C"
    )
).hexdigest()


raw_input_immutability_gate_a4 = (
    train_raw_sha_before_a4
    == train_raw_sha_after_a4
    == EXPECTED_TRAIN_MU_SHA256

    and validation_raw_sha_before_a4
    == validation_raw_sha_after_a4
    == EXPECTED_VALIDATION_MU_SHA256
)


assert raw_input_immutability_gate_a4


# ============================================================
# REPORT
# ============================================================

print()
print(
    "=== NOTEBOOK 05 TRAIN-ONLY ZSCORE FEATURE SPACE ==="
)

print()

print(
    "TRAIN_SHAPE =",
    X_train_zscore.shape,
)

print(
    "VALIDATION_SHAPE =",
    X_validation_zscore.shape,
)

print()

print(
    "SCALER_N_SAMPLES_SEEN =",
    int(
        zscore_scaler.n_samples_seen_
    ),
)

print(
    "MIN_TRAIN_VARIANCE =",
    float(
        scaler_var_before_validation.min()
    ),
)

print(
    "MAX_TRAIN_VARIANCE =",
    float(
        scaler_var_before_validation.max()
    ),
)

print()

print(
    "MAX_ABS_TRAIN_ZSCORE_MEAN =",
    max_abs_train_mean,
)

print(
    "MAX_ABS_TRAIN_ZSCORE_STD_ERROR =",
    max_abs_train_std_error,
)

print()

print(
    "VALIDATION_MEDIAN_ABS_DIMENSION_MEAN =",
    validation_mean_abs_median,
)

print(
    "VALIDATION_MEDIAN_DIMENSION_STD =",
    validation_std_median,
)

print()

print(
    "TRAIN_ZSCORE_SHA256 =",
    train_zscore_sha256,
)

print(
    "VALIDATION_ZSCORE_SHA256 =",
    validation_zscore_sha256,
)

print()

print(
    "SCALER_FIT_COUNT_GATE =",
    scaler_fit_count_gate,
)

print(
    "NONCONSTANT_GATE =",
    nonconstant_gate,
)

print(
    "FINITE_SCALER_GATE =",
    finite_scaler_gate,
)

print(
    "SCALER_IMMUTABILITY_GATE =",
    scaler_immutability_gate,
)

print(
    "TRAIN_STANDARDIZATION_GATE =",
    train_standardization_gate,
)

print(
    "NUMERIC_GATE =",
    numeric_gate_a4,
)

print(
    "RAW_INPUT_IMMUTABILITY_GATE =",
    raw_input_immutability_gate_a4,
)

print()

print(
    "PCA_APPLIED =",
    False,
)

print(
    "LABELS_READ =",
    False,
)

print(
    "KMEANS_EXECUTED_IN_THIS_STEP =",
    False,
)

print(
    "OFFICIAL_DRIVE_ARTIFACTS_MODIFIED =",
    False,
)

print()

print(
    "NOTEBOOK_05_A4A_ZSCORE_FEATURE_SPACE_PASS =",
    (
        scaler_fit_count_gate
        and nonconstant_gate
        and finite_scaler_gate
        and scaler_immutability_gate
        and train_standardization_gate
        and numeric_gate_a4
        and raw_input_immutability_gate_a4
    ),
)


=== NOTEBOOK 05 TRAIN-ONLY ZSCORE FEATURE SPACE ===

TRAIN_SHAPE = (6866, 512)
VALIDATION_SHAPE = (1717, 512)

SCALER_N_SAMPLES_SEEN = 6866
MIN_TRAIN_VARIANCE = 0.0217016556720788
MAX_TRAIN_VARIANCE = 3.376661927847248

MAX_ABS_TRAIN_ZSCORE_MEAN = 1.5679453403308898e-08
MAX_ABS_TRAIN_ZSCORE_STD_ERROR = 1.7427099208155994e-08

VALIDATION_MEDIAN_ABS_DIMENSION_MEAN = 0.014006885929961636
VALIDATION_MEDIAN_DIMENSION_STD = 0.9979964548790616

TRAIN_ZSCORE_SHA256 = 41b220fdb318c4d93a972340acc3ff460c4b9d17920c059a29721454f2b5b1b7
VALIDATION_ZSCORE_SHA256 = aca8724e908cb58537bebd0fd438f819420eafefd5a10a656f76d096653fc50e

SCALER_FIT_COUNT_GATE = True
NONCONSTANT_GATE = True
FINITE_SCALER_GATE = True
SCALER_IMMUTABILITY_GATE = True
TRAIN_STANDARDIZATION_GATE = True
NUMERIC_GATE = True
RAW_INPUT_IMMUTABILITY_GATE = True

PCA_APPLIED = False
LABELS_READ = False
KMEANS_EXECUTED_IN_THIS_STEP = False
OFFICIAL_DRIVE_ARTIFACTS_MODIFIED = False

NOTEBOOK_05_A4A_ZSCORE_FEATURE_SPACE_PASS = True


## STEP 05-A4b — K-means++ nello spazio z-score train-only

La sensitivity analysis pre-registrata utilizza lo spazio standardizzato
definito nello STEP 05-A4a.

Il training utilizza:

$$
\tilde X_{\mathrm{train}}
\in
\mathbb{R}^{6866\times512},
$$

mentre la validation utilizza:

$$
\tilde X_{\mathrm{validation}}
\in
\mathbb{R}^{1717\times512}.
$$

Lo scaler è stato fit esclusivamente sul training e resta congelato.

Il protocollo k-means è identico a quello dell'analisi primaria raw-$\mu$:

$$
k\in\{22,36\},
$$

$$
\mathrm{init}=\texttt{k-means++},
$$

$$
n_{\mathrm{init}}=10,
$$

$$
s_{\mathrm{random}}=123,
$$

$$
n_{\mathrm{iter,max}}=300,
$$

$$
\mathrm{tol}=10^{-4}.
$$

La validation non partecipa al fit e viene assegnata ai centroidi train
congelati.

La standardizzazione modifica la metrica euclidea rispetto allo spazio
raw-$\mu$; pertanto questa analisi è interpretata esclusivamente come
sensitivity analysis.

Non viene applicata PCA e non vengono lette label Gravity Spy.

In [7]:
# ============================================================
# NOTEBOOK 05 — ZSCORE KMEANS SENSITIVITY ANALYSIS
# STEP 05-A4b
#
# k = 22 and k = 36
# n_init = 10
#
# TRAIN-ONLY standardized feature space.
#
# NO LABELS.
# NO PCA.
# NO Drive writes.
# ============================================================

import hashlib
import time

import numpy as np

from sklearn.cluster import KMeans
from threadpoolctl import threadpool_limits


# ============================================================
# FROZEN INPUT HASHES
# ============================================================

train_zscore_sha_before = hashlib.sha256(
    X_train_zscore.astype(
        "<f4",
        copy=False,
    ).tobytes(
        order="C"
    )
).hexdigest()


validation_zscore_sha_before = hashlib.sha256(
    X_validation_zscore.astype(
        "<f4",
        copy=False,
    ).tobytes(
        order="C"
    )
).hexdigest()


assert (
    train_zscore_sha_before
    == train_zscore_sha256
)

assert (
    validation_zscore_sha_before
    == validation_zscore_sha256
)


# ============================================================
# FROZEN KMEANS FACTORY
# ============================================================

def make_zscore_kmeans(k):

    return KMeans(
        n_clusters=k,
        init="k-means++",
        n_init=10,
        max_iter=300,
        tol=1.0e-4,
        verbose=0,
        random_state=123,
        copy_x=True,
        algorithm="lloyd",
    )


# ============================================================
# COMPLETE FIT
#
# Reuses assign_to_frozen_centroids() from STEP 05-A2.
# ============================================================

def run_zscore_kmeans(k):

    estimator = make_zscore_kmeans(
        k
    )


    start_time = time.perf_counter()


    with threadpool_limits(
        limits=1
    ):

        estimator.fit(
            X_train_zscore
        )


    elapsed_seconds = (
        time.perf_counter()
        - start_time
    )


    # --------------------------------------------------------
    # FINAL CENTROIDS
    # --------------------------------------------------------

    centroids = np.asarray(
        estimator.cluster_centers_,
        dtype=np.float32,
        order="C",
    )


    sklearn_train_assignments = np.asarray(
        estimator.labels_,
        dtype=np.int32,
    )


    # --------------------------------------------------------
    # INDEPENDENT ASSIGNMENTS TO FROZEN CENTROIDS
    # --------------------------------------------------------

    (
        train_assignments,
        train_squared_distances,
    ) = assign_to_frozen_centroids(
        X_train_zscore,
        centroids,
    )


    (
        validation_assignments,
        validation_squared_distances,
    ) = assign_to_frozen_centroids(
        X_validation_zscore,
        centroids,
    )


    train_assignment_gate = np.array_equal(
        train_assignments,
        sklearn_train_assignments,
    )


    assert train_assignment_gate


    # --------------------------------------------------------
    # INERTIA
    # --------------------------------------------------------

    sklearn_train_inertia = float(
        estimator.inertia_
    )


    train_inertia = float(
        np.sum(
            train_squared_distances,
            dtype=np.float64,
        )
    )


    validation_inertia = float(
        np.sum(
            validation_squared_distances,
            dtype=np.float64,
        )
    )


    inertia_relative_error = (
        abs(
            train_inertia
            - sklearn_train_inertia
        )
        /
        max(
            abs(
                train_inertia
            ),
            1.0,
        )
    )


    inertia_gate = (
        inertia_relative_error
        < 1.0e-5
    )


    assert inertia_gate


    # --------------------------------------------------------
    # CLUSTER OCCUPANCY
    # --------------------------------------------------------

    train_cluster_sizes = np.bincount(
        train_assignments,
        minlength=k,
    ).astype(
        np.int64,
    )


    validation_cluster_sizes = np.bincount(
        validation_assignments,
        minlength=k,
    ).astype(
        np.int64,
    )


    no_empty_train_cluster_gate = bool(
        np.all(
            train_cluster_sizes > 0
        )
    )


    validation_count_gate = (
        int(
            validation_cluster_sizes.sum()
        )
        == 1717
    )


    assert no_empty_train_cluster_gate
    assert validation_count_gate


    # --------------------------------------------------------
    # NUMERIC / CONVERGENCE
    # --------------------------------------------------------

    numeric_gate = (
        np.isfinite(
            centroids
        ).all()

        and np.isfinite(
            train_squared_distances
        ).all()

        and np.isfinite(
            validation_squared_distances
        ).all()

        and np.isfinite(
            train_inertia
        )

        and np.isfinite(
            validation_inertia
        )
    )


    convergence_gate = (
        1
        <= int(
            estimator.n_iter_
        )
        <= 300
    )


    assert numeric_gate
    assert convergence_gate


    # --------------------------------------------------------
    # HASHES
    # --------------------------------------------------------

    centroid_sha256 = hashlib.sha256(
        centroids.astype(
            "<f4",
            copy=False,
        ).tobytes(
            order="C"
        )
    ).hexdigest()


    train_assignment_sha256 = hashlib.sha256(
        train_assignments.astype(
            "<i4",
            copy=False,
        ).tobytes(
            order="C"
        )
    ).hexdigest()


    validation_assignment_sha256 = hashlib.sha256(
        validation_assignments.astype(
            "<i4",
            copy=False,
        ).tobytes(
            order="C"
        )
    ).hexdigest()


    return {

        "k":
            k,

        "estimator":
            estimator,

        "centroids":
            centroids,

        "train_assignments":
            train_assignments,

        "validation_assignments":
            validation_assignments,

        "train_squared_distances":
            train_squared_distances,

        "validation_squared_distances":
            validation_squared_distances,

        "train_cluster_sizes":
            train_cluster_sizes,

        "validation_cluster_sizes":
            validation_cluster_sizes,

        "iterations":
            int(
                estimator.n_iter_
            ),

        "elapsed_seconds":
            float(
                elapsed_seconds
            ),

        "sklearn_train_inertia":
            sklearn_train_inertia,

        "train_inertia":
            train_inertia,

        "validation_inertia":
            validation_inertia,

        "inertia_relative_error":
            float(
                inertia_relative_error
            ),

        "centroid_sha256":
            centroid_sha256,

        "train_assignment_sha256":
            train_assignment_sha256,

        "validation_assignment_sha256":
            validation_assignment_sha256,

        "train_assignment_gate":
            train_assignment_gate,

        "inertia_gate":
            inertia_gate,

        "no_empty_train_cluster_gate":
            no_empty_train_cluster_gate,

        "validation_count_gate":
            validation_count_gate,

        "numeric_gate":
            numeric_gate,

        "convergence_gate":
            convergence_gate,
    }


# ============================================================
# PRIMARY SENSITIVITY FITS
# ============================================================

print(
    "Running zscore_train_only k=22..."
)

zscore_k22 = run_zscore_kmeans(
    22
)


print(
    "Running zscore_train_only k=36..."
)

zscore_k36 = run_zscore_kmeans(
    36
)


# ============================================================
# REPRODUCIBILITY REFITS
# ============================================================

print(
    "Repeating zscore_train_only k=22..."
)

zscore_k22_repeat = run_zscore_kmeans(
    22
)


print(
    "Repeating zscore_train_only k=36..."
)

zscore_k36_repeat = run_zscore_kmeans(
    36
)


# ============================================================
# EXACT SAME-RUNTIME REPRODUCIBILITY
# ============================================================

def zscore_reproducibility_gate(
    first,
    second,
):

    return (
        first[
            "centroid_sha256"
        ]
        == second[
            "centroid_sha256"
        ]

        and first[
            "train_assignment_sha256"
        ]
        == second[
            "train_assignment_sha256"
        ]

        and first[
            "validation_assignment_sha256"
        ]
        == second[
            "validation_assignment_sha256"
        ]

        and first[
            "iterations"
        ]
        == second[
            "iterations"
        ]

        and first[
            "train_inertia"
        ]
        == second[
            "train_inertia"
        ]

        and first[
            "validation_inertia"
        ]
        == second[
            "validation_inertia"
        ]
    )


zscore_k22_repro_gate = (
    zscore_reproducibility_gate(
        zscore_k22,
        zscore_k22_repeat,
    )
)


zscore_k36_repro_gate = (
    zscore_reproducibility_gate(
        zscore_k36,
        zscore_k36_repeat,
    )
)


assert zscore_k22_repro_gate
assert zscore_k36_repro_gate


# ============================================================
# TRANSFORMED INPUT IMMUTABILITY
# ============================================================

train_zscore_sha_after = hashlib.sha256(
    X_train_zscore.astype(
        "<f4",
        copy=False,
    ).tobytes(
        order="C"
    )
).hexdigest()


validation_zscore_sha_after = hashlib.sha256(
    X_validation_zscore.astype(
        "<f4",
        copy=False,
    ).tobytes(
        order="C"
    )
).hexdigest()


zscore_input_immutability_gate = (
    train_zscore_sha_before
    == train_zscore_sha_after
    == train_zscore_sha256

    and validation_zscore_sha_before
    == validation_zscore_sha_after
    == validation_zscore_sha256
)


assert zscore_input_immutability_gate


# ============================================================
# REPORT
# ============================================================

def report_zscore_run(
    title,
    result,
):

    print()
    print(title)

    print(
        "  k =",
        result["k"],
    )

    print(
        "  n_init = 10"
    )

    print(
        "  iterations_of_selected_solution =",
        result["iterations"],
    )

    print(
        "  elapsed_seconds =",
        result["elapsed_seconds"],
    )

    print(
        "  sklearn_train_inertia =",
        result[
            "sklearn_train_inertia"
        ],
    )

    print(
        "  recomputed_train_inertia =",
        result[
            "train_inertia"
        ],
    )

    print(
        "  validation_inertia =",
        result[
            "validation_inertia"
        ],
    )

    print(
        "  inertia_relative_error =",
        result[
            "inertia_relative_error"
        ],
    )

    print(
        "  min_train_cluster =",
        int(
            result[
                "train_cluster_sizes"
            ].min()
        ),
    )

    print(
        "  max_train_cluster =",
        int(
            result[
                "train_cluster_sizes"
            ].max()
        ),
    )

    print(
        "  empty_validation_clusters =",
        int(
            np.sum(
                result[
                    "validation_cluster_sizes"
                ] == 0
            )
        ),
    )

    print(
        "  centroid_sha256 =",
        result[
            "centroid_sha256"
        ],
    )

    print(
        "  train_assignment_sha256 =",
        result[
            "train_assignment_sha256"
        ],
    )

    print(
        "  validation_assignment_sha256 =",
        result[
            "validation_assignment_sha256"
        ],
    )


print()
print(
    "=== NOTEBOOK 05 ZSCORE KMEANS SENSITIVITY ==="
)


report_zscore_run(
    "ZSCORE_K22",
    zscore_k22,
)


report_zscore_run(
    "ZSCORE_K36",
    zscore_k36,
)


print()

print(
    "ZSCORE_K22_REPRODUCIBILITY_GATE =",
    zscore_k22_repro_gate,
)

print(
    "ZSCORE_K36_REPRODUCIBILITY_GATE =",
    zscore_k36_repro_gate,
)

print(
    "ZSCORE_INPUT_IMMUTABILITY_GATE =",
    zscore_input_immutability_gate,
)

print()

print(
    "LABELS_READ =",
    False,
)

print(
    "VALIDATION_REFIT =",
    False,
)

print(
    "PCA_APPLIED =",
    False,
)

print(
    "STANDARDIZATION_APPLIED =",
    True,
)

print(
    "CLUSTERING_PUBLISHED =",
    False,
)

print(
    "OFFICIAL_DRIVE_ARTIFACTS_MODIFIED =",
    False,
)

print()

print(
    "NOTEBOOK_05_A4B_ZSCORE_KMEANS_PASS =",
    (
        zscore_k22[
            "train_assignment_gate"
        ]

        and zscore_k22[
            "inertia_gate"
        ]

        and zscore_k22[
            "no_empty_train_cluster_gate"
        ]

        and zscore_k22[
            "validation_count_gate"
        ]

        and zscore_k22[
            "numeric_gate"
        ]

        and zscore_k22[
            "convergence_gate"
        ]

        and zscore_k36[
            "train_assignment_gate"
        ]

        and zscore_k36[
            "inertia_gate"
        ]

        and zscore_k36[
            "no_empty_train_cluster_gate"
        ]

        and zscore_k36[
            "validation_count_gate"
        ]

        and zscore_k36[
            "numeric_gate"
        ]

        and zscore_k36[
            "convergence_gate"
        ]

        and zscore_k22_repro_gate
        and zscore_k36_repro_gate
        and zscore_input_immutability_gate
    ),
)

Running zscore_train_only k=22...
Running zscore_train_only k=36...
Repeating zscore_train_only k=22...
Repeating zscore_train_only k=36...

=== NOTEBOOK 05 ZSCORE KMEANS SENSITIVITY ===

ZSCORE_K22
  k = 22
  n_init = 10
  iterations_of_selected_solution = 31
  elapsed_seconds = 3.4318553329994756
  sklearn_train_inertia = 1619279.625
  recomputed_train_inertia = 1619277.5761127267
  validation_inertia = 406912.3988701601
  inertia_relative_error = 1.2653094833903138e-06
  min_train_cluster = 39
  max_train_cluster = 1607
  empty_validation_clusters = 0
  centroid_sha256 = 79e5b3da9ccccbad484f54e9c29ceb607a46b1067943e53745c840f6a671649c
  train_assignment_sha256 = 9c3f78c8fbec8614984e811bbed3fd9dc4333c4b94c49efecc00de13e7a7ca8a
  validation_assignment_sha256 = af5a466aaefcf15b17d60d6289ed59ecb3a03773187573ad73a7cbcad7f1c23e

ZSCORE_K36
  k = 36
  n_init = 10
  iterations_of_selected_solution = 19
  elapsed_seconds = 4.302149174000078
  sklearn_train_inertia = 1365278.125
  recomputed_

## STEP 05-A4c — Diagnostiche interne della sensitivity z-score

Il clustering ottenuto nello spazio standardizzato viene valutato con le
stesse diagnostiche interne utilizzate per lo spazio raw-$\mu$.

Vengono calcolati:

$$
\mathrm{Silhouette},
$$

$$
\mathrm{Davies\text{-}Bouldin},
$$

occupazione dei cluster e inertia media per evento.

I valori assoluti dell'inertia non vengono confrontati fra raw-$\mu$ e
z-score, perché la standardizzazione modifica la metrica euclidea.

Viene inoltre misurata, senza utilizzare le label Gravity Spy, la stabilità
della partizione rispetto alla trasformazione delle feature.

Per questo si confrontano gli assignment raw-$\mu$ e z-score mediante:

$$
\mathrm{ARI}_{\mathrm{raw,zscore}}
$$

e

$$
\mathrm{NMI}_{\mathrm{raw,zscore}}.
$$

Queste quantità confrontano due partizioni unsupervised dello stesso insieme
di eventi e non costituiscono una valutazione rispetto alle classi note.

Le label Gravity Spy restano non lette.

In [8]:
# ============================================================
# NOTEBOOK 05 — ZSCORE INTERNAL DIAGNOSTICS
# STEP 05-A4c
#
# LABEL-FREE comparison:
#   raw_mu vs zscore_train_only
#
# NO Gravity Spy labels.
# NO Drive writes.
# ============================================================

import numpy as np

from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
)


# ============================================================
# INTERNAL DIAGNOSTICS
#
# Reuses compute_internal_diagnostics() from STEP 05-A3.
# ============================================================

print(
    "Computing zscore k=22 train diagnostics..."
)

zscore_k22_train_internal = (
    compute_internal_diagnostics(
        X_train_zscore,
        zscore_k22[
            "train_assignments"
        ],
        zscore_k22[
            "train_inertia"
        ],
        name="zscore_k22_train",
    )
)


print(
    "Computing zscore k=22 validation diagnostics..."
)

zscore_k22_validation_internal = (
    compute_internal_diagnostics(
        X_validation_zscore,
        zscore_k22[
            "validation_assignments"
        ],
        zscore_k22[
            "validation_inertia"
        ],
        name="zscore_k22_validation",
    )
)


print(
    "Computing zscore k=36 train diagnostics..."
)

zscore_k36_train_internal = (
    compute_internal_diagnostics(
        X_train_zscore,
        zscore_k36[
            "train_assignments"
        ],
        zscore_k36[
            "train_inertia"
        ],
        name="zscore_k36_train",
    )
)


print(
    "Computing zscore k=36 validation diagnostics..."
)

zscore_k36_validation_internal = (
    compute_internal_diagnostics(
        X_validation_zscore,
        zscore_k36[
            "validation_assignments"
        ],
        zscore_k36[
            "validation_inertia"
        ],
        name="zscore_k36_validation",
    )
)


# ============================================================
# RAW-vs-ZSCORE PARTITION STABILITY
#
# No semantic labels are used here.
# ============================================================

raw_vs_zscore_k22_train_ari = float(
    adjusted_rand_score(
        raw_mu_k22[
            "train_assignments"
        ],
        zscore_k22[
            "train_assignments"
        ],
    )
)


raw_vs_zscore_k22_validation_ari = float(
    adjusted_rand_score(
        raw_mu_k22[
            "validation_assignments"
        ],
        zscore_k22[
            "validation_assignments"
        ],
    )
)


raw_vs_zscore_k36_train_ari = float(
    adjusted_rand_score(
        raw_mu_k36[
            "train_assignments"
        ],
        zscore_k36[
            "train_assignments"
        ],
    )
)


raw_vs_zscore_k36_validation_ari = float(
    adjusted_rand_score(
        raw_mu_k36[
            "validation_assignments"
        ],
        zscore_k36[
            "validation_assignments"
        ],
    )
)


raw_vs_zscore_k22_train_nmi = float(
    normalized_mutual_info_score(
        raw_mu_k22[
            "train_assignments"
        ],
        zscore_k22[
            "train_assignments"
        ],
        average_method="arithmetic",
    )
)


raw_vs_zscore_k22_validation_nmi = float(
    normalized_mutual_info_score(
        raw_mu_k22[
            "validation_assignments"
        ],
        zscore_k22[
            "validation_assignments"
        ],
        average_method="arithmetic",
    )
)


raw_vs_zscore_k36_train_nmi = float(
    normalized_mutual_info_score(
        raw_mu_k36[
            "train_assignments"
        ],
        zscore_k36[
            "train_assignments"
        ],
        average_method="arithmetic",
    )
)


raw_vs_zscore_k36_validation_nmi = float(
    normalized_mutual_info_score(
        raw_mu_k36[
            "validation_assignments"
        ],
        zscore_k36[
            "validation_assignments"
        ],
        average_method="arithmetic",
    )
)


# ============================================================
# GENERALIZATION GAPS
# ============================================================

zscore_k22_inertia_gap_relative = (
    zscore_k22_validation_internal[
        "inertia_per_event"
    ]
    /
    zscore_k22_train_internal[
        "inertia_per_event"
    ]
    - 1.0
)


zscore_k36_inertia_gap_relative = (
    zscore_k36_validation_internal[
        "inertia_per_event"
    ]
    /
    zscore_k36_train_internal[
        "inertia_per_event"
    ]
    - 1.0
)


# ============================================================
# REPORT
# ============================================================

print()
print(
    "=== NOTEBOOK 05 ZSCORE INTERNAL DIAGNOSTICS ==="
)


report_internal(
    zscore_k22_train_internal
)

report_internal(
    zscore_k22_validation_internal
)

report_internal(
    zscore_k36_train_internal
)

report_internal(
    zscore_k36_validation_internal
)


print()
print(
    "=== RAW-MU vs ZSCORE PARTITION STABILITY ==="
)

print()

print(
    "K22_TRAIN_ARI =",
    raw_vs_zscore_k22_train_ari,
)

print(
    "K22_VALIDATION_ARI =",
    raw_vs_zscore_k22_validation_ari,
)

print(
    "K22_TRAIN_NMI =",
    raw_vs_zscore_k22_train_nmi,
)

print(
    "K22_VALIDATION_NMI =",
    raw_vs_zscore_k22_validation_nmi,
)

print()

print(
    "K36_TRAIN_ARI =",
    raw_vs_zscore_k36_train_ari,
)

print(
    "K36_VALIDATION_ARI =",
    raw_vs_zscore_k36_validation_ari,
)

print(
    "K36_TRAIN_NMI =",
    raw_vs_zscore_k36_train_nmi,
)

print(
    "K36_VALIDATION_NMI =",
    raw_vs_zscore_k36_validation_nmi,
)

print()

print(
    "ZSCORE_K22_INERTIA_PER_EVENT_RELATIVE_GAP =",
    zscore_k22_inertia_gap_relative,
)

print(
    "ZSCORE_K36_INERTIA_PER_EVENT_RELATIVE_GAP =",
    zscore_k36_inertia_gap_relative,
)


# ============================================================
# HARD GATES
# ============================================================

partition_stability_numeric_gate = all(
    np.isfinite(
        [
            raw_vs_zscore_k22_train_ari,
            raw_vs_zscore_k22_validation_ari,
            raw_vs_zscore_k36_train_ari,
            raw_vs_zscore_k36_validation_ari,
            raw_vs_zscore_k22_train_nmi,
            raw_vs_zscore_k22_validation_nmi,
            raw_vs_zscore_k36_train_nmi,
            raw_vs_zscore_k36_validation_nmi,
        ]
    )
)


internal_zscore_gate = all([
    zscore_k22_train_internal[
        "numeric_gate"
    ],

    zscore_k22_validation_internal[
        "numeric_gate"
    ],

    zscore_k36_train_internal[
        "numeric_gate"
    ],

    zscore_k36_validation_internal[
        "numeric_gate"
    ],

    partition_stability_numeric_gate,
])


assert internal_zscore_gate


print()

print(
    "PARTITION_STABILITY_NUMERIC_GATE =",
    partition_stability_numeric_gate,
)

print(
    "LABELS_READ =",
    False,
)

print(
    "PCA_APPLIED =",
    False,
)

print(
    "OFFICIAL_DRIVE_ARTIFACTS_MODIFIED =",
    False,
)

print()

print(
    "NOTEBOOK_05_A4C_INTERNAL_SENSITIVITY_PASS =",
    internal_zscore_gate,
)

Computing zscore k=22 train diagnostics...
Computing zscore k=22 validation diagnostics...
Computing zscore k=36 train diagnostics...
Computing zscore k=36 validation diagnostics...

=== NOTEBOOK 05 ZSCORE INTERNAL DIAGNOSTICS ===

zscore_k22_train
  n_samples = 6866
  n_clusters = 22
  inertia_per_event = 235.84001982416643
  silhouette = 0.18102937936782837
  silhouette_seconds = 1.4114193830009754
  davies_bouldin = 1.800744401251223
  cluster_size_quantiles[min,q25,median,q75,max] = [39.0, 116.5, 205.0, 367.25, 1607.0]
  effective_cluster_count = 14.238095239272287

zscore_k22_validation
  n_samples = 1717
  n_clusters = 22
  inertia_per_event = 236.99033131634252
  silhouette = 0.18585234880447388
  silhouette_seconds = 0.1894677660002344
  davies_bouldin = 1.7648693434806793
  cluster_size_quantiles[min,q25,median,q75,max] = [8.0, 22.25, 48.5, 94.75, 385.0]
  effective_cluster_count = 14.170087581491117

zscore_k36_train
  n_samples = 6866
  n_clusters = 36
  inertia_per_event = 

## STEP 05-A5 — Pubblicazione del clustering k-means label-blind

Le analisi k-means del VAE `sakai224_full` sono pubblicate nell'artefatto
persistente e congelato `sakai224_full_kmeans_labelblind_v1`, prima di
qualsiasi valutazione con le label Gravity Spy.

L'artefatto contiene due rami distinti.

### Analisi primaria

$$
\mathrm{raw}\text{-}\mu
$$

con:

$$
k=22
\qquad\text{e}\qquad
k=36.
$$

### Sensitivity analysis

$$
\mathrm{zscore\ train\text{-}only}
$$

con gli stessi:

$$
k=22
\qquad\text{e}\qquad
k=36.
$$

Per ogni clustering vengono congelati:

- centroidi train;
- assignment train;
- assignment validation;
- distanze quadratiche dal centroide assegnato;
- dimensioni dei cluster;
- diagnostiche interne;
- identità SHA-256.

Per la sensitivity vengono inoltre congelati:

- media train dello scaler;
- varianza train;
- scale train;
- matrici standardizzate esatte utilizzate dal clustering.

Le row-map vengono copiate dall'artefatto canonico degli embedding per
preservare l'identità delle righe senza introdurre label.

L'artefatto viene costruito inizialmente in una directory `.partial`,
verificato integralmente tramite SHA-256 e soltanto dopo promosso al nome
definitivo.

Le label Gravity Spy non vengono lette e non vengono incluse.

La valutazione esterna viene deliberatamente rinviata fino al congelamento
della pipeline principale:

$$
\mathrm{VAE}
\rightarrow
\mathrm{IIC}
\rightarrow
\mathrm{over\text{-}clustering}
\rightarrow
K=5
\rightarrow
\mathrm{spectral\ clustering}.
$$

Il k-means resta una diagnostica PROJECT delle feature VAE e non viene
presentato come classificatore finale di Sakai.

In [9]:
# ============================================================
# NOTEBOOK 05 — PUBLISH LABEL-BLIND KMEANS ARTIFACT
# STEP 05-A5
#
# WRITES a NEW persistent artifact on Drive.
#
# Does NOT modify:
#   - VAE checkpoints
#   - canonical embedding artifact
#   - Notebook 04 artifacts
#
# NO Gravity Spy labels.
# ============================================================

from pathlib import Path

import hashlib
import json
import os
import platform
import shutil

import numpy as np
import scipy
import sklearn
import threadpoolctl


# ============================================================
# FROZEN IDENTITIES
# ============================================================

EXPECTED_CORRECTED_CONFIG_SHA256 = (
    "f43a89b318bf22f03a63d2a7e5a995917"
    "e97f160c7d8eccbb19fade9a6457d29"
)

assert (
    KMEANS_CONFIG_SHA256
    == EXPECTED_CORRECTED_CONFIG_SHA256
)

assert (
    KMEANS_CONFIG[
        "main_scientific_goal"
    ]
    == "sakai224_full_vs_Sakai"
)

assert (
    KMEANS_CONFIG[
        "primary_feature_space"
    ][
        "name"
    ]
    == "raw_mu"
)

assert (
    KMEANS_CONFIG[
        "sensitivity_feature_space"
    ][
        "name"
    ]
    == "zscore_train_only"
)


# ============================================================
# OUTPUT PATHS
# ============================================================

KMEANS_ARTIFACT_ID = (
    "sakai224_full_kmeans_labelblind_v1"
)

KMEANS_PARENT_DIR = (
    SAKAI224_ROOT
    / "kmeans"
)

KMEANS_FINAL_DIR = (
    KMEANS_PARENT_DIR
    / KMEANS_ARTIFACT_ID
)

KMEANS_PARTIAL_DIR = (
    KMEANS_PARENT_DIR
    / (
        KMEANS_ARTIFACT_ID
        + ".partial"
    )
)


KMEANS_PARENT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# Refuse accidental overwrite.
assert not KMEANS_FINAL_DIR.exists(), (
    "Final artifact already exists: "
    f"{KMEANS_FINAL_DIR}"
)

assert not KMEANS_PARTIAL_DIR.exists(), (
    "Partial artifact already exists: "
    f"{KMEANS_PARTIAL_DIR}"
)


KMEANS_PARTIAL_DIR.mkdir(
    parents=False,
    exist_ok=False,
)


# ============================================================
# I/O HELPERS
# ============================================================

def sha256_path(
    path,
    chunk_size=8 * 1024 * 1024,
):

    digest = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as handle:

        while True:

            block = handle.read(
                chunk_size
            )

            if not block:
                break

            digest.update(
                block
            )

    return digest.hexdigest()


def atomic_write_bytes(
    path,
    payload,
):

    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temp_path = Path(
        str(path) + ".tmp"
    )


    with temp_path.open(
        "wb"
    ) as handle:

        handle.write(
            payload
        )

        handle.flush()

        os.fsync(
            handle.fileno()
        )


    os.replace(
        temp_path,
        path,
    )


def write_array(
    path,
    array,
    dtype,
):

    encoded = np.asarray(
        array,
        dtype=np.dtype(
            dtype
        ),
        order="C",
    )


    atomic_write_bytes(
        path,
        encoded.tobytes(
            order="C"
        ),
    )


    return sha256_path(
        path
    )


def write_json(
    path,
    obj,
):

    payload = json.dumps(
        obj,
        sort_keys=True,
        indent=2,
        ensure_ascii=True,
    ).encode(
        "utf-8"
    )


    atomic_write_bytes(
        path,
        payload,
    )


    return sha256_path(
        path
    )


# ============================================================
# HUMAN-INDEPENDENT SUMMARY HELPERS
# ============================================================

def internal_summary(
    result,
):

    return {

        "n_samples":
            int(
                result[
                    "n_samples"
                ]
            ),

        "n_features":
            int(
                result[
                    "n_features"
                ]
            ),

        "n_clusters":
            int(
                result[
                    "n_clusters"
                ]
            ),

        "inertia":
            float(
                result[
                    "inertia"
                ]
            ),

        "inertia_per_event":
            float(
                result[
                    "inertia_per_event"
                ]
            ),

        "silhouette":
            float(
                result[
                    "silhouette"
                ]
            ),

        "davies_bouldin":
            float(
                result[
                    "davies_bouldin"
                ]
            ),

        "cluster_size_quantiles":
            [
                float(x)
                for x
                in result[
                    "occupancy_quantiles"
                ]
            ],

        "effective_cluster_count":
            float(
                result[
                    "effective_cluster_count"
                ]
            ),
    }


def publish_clustering(
    *,
    base_dir,
    result,
    train_internal,
    validation_internal,
    feature_space,
):

    k = int(
        result[
            "k"
        ]
    )


    run_dir = (
        Path(base_dir)
        / f"k{k}"
    )


    run_dir.mkdir(
        parents=True,
        exist_ok=False,
    )


    # --------------------------------------------------------
    # MACHINE-READABLE ARRAYS
    # --------------------------------------------------------

    centroid_file_sha = write_array(
        run_dir
        / "centroids_f32.bin",
        result[
            "centroids"
        ],
        "<f4",
    )


    train_assignment_file_sha = write_array(
        run_dir
        / "train_assignments_i32.bin",
        result[
            "train_assignments"
        ],
        "<i4",
    )


    validation_assignment_file_sha = write_array(
        run_dir
        / "validation_assignments_i32.bin",
        result[
            "validation_assignments"
        ],
        "<i4",
    )


    train_distance_file_sha = write_array(
        run_dir
        / "train_squared_distances_f64.bin",
        result[
            "train_squared_distances"
        ],
        "<f8",
    )


    validation_distance_file_sha = write_array(
        run_dir
        / "validation_squared_distances_f64.bin",
        result[
            "validation_squared_distances"
        ],
        "<f8",
    )


    train_size_file_sha = write_array(
        run_dir
        / "train_cluster_sizes_i64.bin",
        result[
            "train_cluster_sizes"
        ],
        "<i8",
    )


    validation_size_file_sha = write_array(
        run_dir
        / "validation_cluster_sizes_i64.bin",
        result[
            "validation_cluster_sizes"
        ],
        "<i8",
    )


    # --------------------------------------------------------
    # ASSERT AGAINST ALREADY OBSERVED IDENTITIES
    # --------------------------------------------------------

    assert (
        centroid_file_sha
        == result[
            "centroid_sha256"
        ]
    )

    assert (
        train_assignment_file_sha
        == result[
            "train_assignment_sha256"
        ]
    )

    assert (
        validation_assignment_file_sha
        == result[
            "validation_assignment_sha256"
        ]
    )


    # --------------------------------------------------------
    # RUN SUMMARY
    # --------------------------------------------------------

    summary = {

        "feature_space":
            feature_space,

        "k":
            k,

        "init":
            "k-means++",

        "n_init":
            10,

        "random_state":
            123,

        "max_iter":
            300,

        "tol":
            1.0e-4,

        "algorithm":
            "lloyd",

        "fit_split":
            "train",

        "validation_policy":
            "nearest_frozen_train_centroid",

        "iterations_of_selected_solution":
            int(
                result[
                    "iterations"
                ]
            ),

        "sklearn_train_inertia":
            float(
                result[
                    "sklearn_train_inertia"
                ]
            ),

        "recomputed_train_inertia":
            float(
                result[
                    "train_inertia"
                ]
            ),

        "validation_inertia":
            float(
                result[
                    "validation_inertia"
                ]
            ),

        "inertia_relative_error":
            float(
                result[
                    "inertia_relative_error"
                ]
            ),

        "train_cluster_min":
            int(
                result[
                    "train_cluster_sizes"
                ].min()
            ),

        "train_cluster_max":
            int(
                result[
                    "train_cluster_sizes"
                ].max()
            ),

        "validation_empty_clusters":
            int(
                np.sum(
                    result[
                        "validation_cluster_sizes"
                    ]
                    == 0
                )
            ),

        "train_internal":
            internal_summary(
                train_internal
            ),

        "validation_internal":
            internal_summary(
                validation_internal
            ),

        "files": {

            "centroids_f32.bin":
                centroid_file_sha,

            "train_assignments_i32.bin":
                train_assignment_file_sha,

            "validation_assignments_i32.bin":
                validation_assignment_file_sha,

            "train_squared_distances_f64.bin":
                train_distance_file_sha,

            "validation_squared_distances_f64.bin":
                validation_distance_file_sha,

            "train_cluster_sizes_i64.bin":
                train_size_file_sha,

            "validation_cluster_sizes_i64.bin":
                validation_size_file_sha,
        },

        "labels_used":
            False,
    }


    summary_sha = write_json(
        run_dir
        / "summary.json",
        summary,
    )


    return {
        "directory":
            str(
                run_dir.relative_to(
                    KMEANS_PARTIAL_DIR
                )
            ),

        "summary":
            summary,

        "summary_sha256":
            summary_sha,
    }


# ============================================================
# COPY LABEL-FREE ROW MAPS
# ============================================================

shutil.copy2(
    TRAIN_ROW_MAP_PATH,
    KMEANS_PARTIAL_DIR
    / TRAIN_ROW_MAP_NAME,
)

shutil.copy2(
    VALIDATION_ROW_MAP_PATH,
    KMEANS_PARTIAL_DIR
    / VALIDATION_ROW_MAP_NAME,
)


assert (
    sha256_path(
        KMEANS_PARTIAL_DIR
        / TRAIN_ROW_MAP_NAME
    )
    == calculated_hashes[
        TRAIN_ROW_MAP_NAME
    ]
)

assert (
    sha256_path(
        KMEANS_PARTIAL_DIR
        / VALIDATION_ROW_MAP_NAME
    )
    == calculated_hashes[
        VALIDATION_ROW_MAP_NAME
    ]
)


# ============================================================
# FROZEN CONFIG
#
# Write EXACT canonical bytes so file SHA equals the
# corrected config SHA frozen in STEP 05-A1.
# ============================================================

atomic_write_bytes(
    KMEANS_PARTIAL_DIR
    / "kmeans_config_v2.json",
    KMEANS_CONFIG_JSON.encode(
        "utf-8"
    ),
)


assert (
    sha256_path(
        KMEANS_PARTIAL_DIR
        / "kmeans_config_v2.json"
    )
    == EXPECTED_CORRECTED_CONFIG_SHA256
)


# ============================================================
# PRIMARY RAW-MU BRANCH
# ============================================================

raw_dir = (
    KMEANS_PARTIAL_DIR
    / "raw_mu"
)


raw_k22_publication = publish_clustering(
    base_dir=raw_dir,
    result=raw_mu_k22,
    train_internal=raw_mu_k22_train_internal,
    validation_internal=raw_mu_k22_validation_internal,
    feature_space="raw_mu",
)


raw_k36_publication = publish_clustering(
    base_dir=raw_dir,
    result=raw_mu_k36,
    train_internal=raw_mu_k36_train_internal,
    validation_internal=raw_mu_k36_validation_internal,
    feature_space="raw_mu",
)


# ============================================================
# Z-SCORE SENSITIVITY BRANCH
# ============================================================

zscore_dir = (
    KMEANS_PARTIAL_DIR
    / "zscore_train_only"
)


zscore_dir.mkdir(
    parents=True,
    exist_ok=False,
)


# Exact standardized matrices used for clustering.
published_train_zscore_sha = write_array(
    zscore_dir
    / "train_features_f32.bin",
    X_train_zscore,
    "<f4",
)


published_validation_zscore_sha = write_array(
    zscore_dir
    / "validation_features_f32.bin",
    X_validation_zscore,
    "<f4",
)


assert (
    published_train_zscore_sha
    == train_zscore_sha256
)

assert (
    published_validation_zscore_sha
    == validation_zscore_sha256
)


# Exact train-only scaler.
scaler_mean_sha = write_array(
    zscore_dir
    / "scaler_mean_f64.bin",
    scaler_mean_before_validation,
    "<f8",
)


scaler_var_sha = write_array(
    zscore_dir
    / "scaler_var_f64.bin",
    scaler_var_before_validation,
    "<f8",
)


scaler_scale_sha = write_array(
    zscore_dir
    / "scaler_scale_f64.bin",
    scaler_scale_before_validation,
    "<f8",
)


scaler_summary = {

    "fit_split":
        "train",

    "n_samples_seen":
        int(
            zscore_scaler.n_samples_seen_
        ),

    "n_features":
        512,

    "variance_definition":
        "population_ddof0",

    "validation_refit":
        False,

    "max_abs_train_transformed_mean":
        float(
            max_abs_train_mean
        ),

    "max_abs_train_transformed_std_error":
        float(
            max_abs_train_std_error
        ),

    "train_features_sha256":
        published_train_zscore_sha,

    "validation_features_sha256":
        published_validation_zscore_sha,

    "scaler_mean_sha256":
        scaler_mean_sha,

    "scaler_var_sha256":
        scaler_var_sha,

    "scaler_scale_sha256":
        scaler_scale_sha,

    "labels_used":
        False,
}


write_json(
    zscore_dir
    / "scaler_summary.json",
    scaler_summary,
)


zscore_k22_publication = publish_clustering(
    base_dir=zscore_dir,
    result=zscore_k22,
    train_internal=zscore_k22_train_internal,
    validation_internal=zscore_k22_validation_internal,
    feature_space="zscore_train_only",
)


zscore_k36_publication = publish_clustering(
    base_dir=zscore_dir,
    result=zscore_k36,
    train_internal=zscore_k36_train_internal,
    validation_internal=zscore_k36_validation_internal,
    feature_space="zscore_train_only",
)


# ============================================================
# RAW-vs-ZSCORE LABEL-FREE STABILITY
# ============================================================

partition_stability = {

    "interpretation":
        (
            "comparison between two unsupervised "
            "partitions; no Gravity Spy labels"
        ),

    "k22": {

        "train_ari":
            float(
                raw_vs_zscore_k22_train_ari
            ),

        "validation_ari":
            float(
                raw_vs_zscore_k22_validation_ari
            ),

        "train_nmi_arithmetic":
            float(
                raw_vs_zscore_k22_train_nmi
            ),

        "validation_nmi_arithmetic":
            float(
                raw_vs_zscore_k22_validation_nmi
            ),
    },

    "k36": {

        "train_ari":
            float(
                raw_vs_zscore_k36_train_ari
            ),

        "validation_ari":
            float(
                raw_vs_zscore_k36_validation_ari
            ),

        "train_nmi_arithmetic":
            float(
                raw_vs_zscore_k36_train_nmi
            ),

        "validation_nmi_arithmetic":
            float(
                raw_vs_zscore_k36_validation_nmi
            ),
    },

    "labels_used":
        False,
}


write_json(
    KMEANS_PARTIAL_DIR
    / "raw_vs_zscore_partition_stability.json",
    partition_stability,
)


# ============================================================
# TOP-LEVEL METADATA
# ============================================================

metadata = {

    "artifact_format_version":
        1,

    "artifact_id":
        KMEANS_ARTIFACT_ID,

    "experiment":
        "sakai224_full",

    "scientific_role":
        (
            "PROJECT label-blind k-means diagnostic "
            "of epoch-100 VAE embeddings"
        ),

    "main_scientific_goal":
        "sakai224_full_vs_Sakai",

    "is_Sakai_final_classifier_replication":
        False,

    "source_embedding_artifact":
        EXPECTED_ARTIFACT_ID,

    "source_train_mu_sha256":
        EXPECTED_TRAIN_MU_SHA256,

    "source_validation_mu_sha256":
        EXPECTED_VALIDATION_MU_SHA256,

    "source_train_row_map_sha256":
        calculated_hashes[
            TRAIN_ROW_MAP_NAME
        ],

    "source_validation_row_map_sha256":
        calculated_hashes[
            VALIDATION_ROW_MAP_NAME
        ],

    "source_embedding_metadata_sha256":
        calculated_hashes[
            METADATA_NAME
        ],

    "kmeans_config_sha256":
        EXPECTED_CORRECTED_CONFIG_SHA256,

    "superseded_config_sha256":
        SUPERSEDED_KMEANS_CONFIG_SHA256,

    "primary_feature_space":
        "raw_mu",

    "sensitivity_feature_space":
        "zscore_train_only",

    "k_values": [
        22,
        36,
    ],

    "labels_used":
        False,

    "external_metrics_computed":
        False,

    "gravity_spy_contingency_computed":
        False,

    "pca_applied":
        False,

    "validation_refit":
        False,

    "external_evaluation_deferred_until_after_IIC_spectral":
        True,

    "future_primary_pipeline": [
        "VAE",
        "mu",
        "IIC",
        "over_clustering",
        "K5_classifiers",
        "spectral_clustering",
        "posterior_label_evaluation",
    ],

    "runtime": {

        "python":
            platform.python_version(),

        "numpy":
            np.__version__,

        "scipy":
            scipy.__version__,

        "scikit_learn":
            sklearn.__version__,

        "threadpoolctl":
            threadpoolctl.__version__,

        "numeric_thread_limit":
            1,
    },

    "raw_mu": {

        "k22":
            raw_k22_publication,

        "k36":
            raw_k36_publication,
    },

    "zscore_train_only": {

        "train_features_sha256":
            published_train_zscore_sha,

        "validation_features_sha256":
            published_validation_zscore_sha,

        "k22":
            zscore_k22_publication,

        "k36":
            zscore_k36_publication,
    },

    "labels_policy":
        (
            "Gravity Spy labels intentionally deferred "
            "until the complete unsupervised pipeline "
            "has been frozen"
        ),
}


write_json(
    KMEANS_PARTIAL_DIR
    / "metadata.json",
    metadata,
)


# ============================================================
# BUILD PACKAGE SHA256SUMS
# ============================================================

package_files = sorted(
    [
        path
        for path
        in KMEANS_PARTIAL_DIR.rglob(
            "*"
        )
        if (
            path.is_file()
            and path.name
            != "SHA256SUMS.txt"
            and not path.name.endswith(
                ".tmp"
            )
        )
    ],
    key=lambda p: str(
        p.relative_to(
            KMEANS_PARTIAL_DIR
        )
    ),
)


checksum_lines = []


for path in package_files:

    relative = path.relative_to(
        KMEANS_PARTIAL_DIR
    )

    digest = sha256_path(
        path
    )

    checksum_lines.append(
        f"{digest}  {relative.as_posix()}"
    )


checksum_payload = (
    "\n".join(
        checksum_lines
    )
    + "\n"
).encode(
    "ascii"
)


atomic_write_bytes(
    KMEANS_PARTIAL_DIR
    / "SHA256SUMS.txt",
    checksum_payload,
)


SHA256SUMS_SHA256 = sha256_path(
    KMEANS_PARTIAL_DIR
    / "SHA256SUMS.txt"
)


# ============================================================
# VERIFY PARTIAL PACKAGE BEFORE PROMOTION
# ============================================================

def verify_package(
    directory,
):

    directory = Path(
        directory
    )

    checksum_path = (
        directory
        / "SHA256SUMS.txt"
    )


    records = []


    with checksum_path.open(
        "r",
        encoding="ascii",
    ) as handle:

        for line in handle:

            line = line.rstrip(
                "\n"
            )

            if not line:
                continue


            digest, relative = line.split(
                "  ",
                maxsplit=1,
            )


            records.append(
                (
                    digest,
                    relative,
                )
            )


    all_hashes_ok = True


    for expected_digest, relative in records:

        actual_digest = sha256_path(
            directory
            / relative
        )


        if (
            actual_digest
            != expected_digest
        ):

            all_hashes_ok = False

            print(
                "HASH MISMATCH:",
                relative,
            )


    temp_files = list(
        directory.rglob(
            "*.tmp"
        )
    )


    return {
        "records":
            records,

        "all_hashes_ok":
            all_hashes_ok,

        "temp_files":
            temp_files,
    }


partial_verification = verify_package(
    KMEANS_PARTIAL_DIR
)


assert partial_verification[
    "all_hashes_ok"
]

assert len(
    partial_verification[
        "temp_files"
    ]
) == 0


# ============================================================
# PROMOTE PARTIAL -> FINAL
#
# Same Drive parent; no overwrite allowed.
# ============================================================

os.rename(
    KMEANS_PARTIAL_DIR,
    KMEANS_FINAL_DIR,
)


assert KMEANS_FINAL_DIR.is_dir()
assert not KMEANS_PARTIAL_DIR.exists()


# ============================================================
# FINAL IN-SITU VERIFICATION
# ============================================================

final_verification = verify_package(
    KMEANS_FINAL_DIR
)


assert final_verification[
    "all_hashes_ok"
]

assert len(
    final_verification[
        "temp_files"
    ]
) == 0


assert (
    sha256_path(
        KMEANS_FINAL_DIR
        / "SHA256SUMS.txt"
    )
    == SHA256SUMS_SHA256
)


# ============================================================
# FINAL INVENTORY
# ============================================================

final_files = sorted(
    [
        path
        for path
        in KMEANS_FINAL_DIR.rglob(
            "*"
        )
        if path.is_file()
    ]
)


final_total_bytes = sum(
    path.stat().st_size
    for path
    in final_files
)


# ============================================================
# FINAL REPORT
# ============================================================

print()
print(
    "=== NOTEBOOK 05 LABEL-BLIND KMEANS PUBLICATION ==="
)

print()

print(
    "ARTIFACT_ID =",
    KMEANS_ARTIFACT_ID,
)

print(
    "FINAL_DIRECTORY =",
    KMEANS_FINAL_DIR,
)

print()

print(
    "FILE_COUNT =",
    len(
        final_files
    ),
)

print(
    "TOTAL_BYTES =",
    final_total_bytes,
)

print()

print(
    "KMEANS_CONFIG_SHA256 =",
    EXPECTED_CORRECTED_CONFIG_SHA256,
)

print(
    "SHA256SUMS_SHA256 =",
    SHA256SUMS_SHA256,
)

print()

print(
    "RAW_K22_CENTROID_SHA256 =",
    raw_mu_k22[
        "centroid_sha256"
    ],
)

print(
    "RAW_K22_TRAIN_ASSIGNMENT_SHA256 =",
    raw_mu_k22[
        "train_assignment_sha256"
    ],
)

print(
    "RAW_K22_VALIDATION_ASSIGNMENT_SHA256 =",
    raw_mu_k22[
        "validation_assignment_sha256"
    ],
)

print()

print(
    "RAW_K36_CENTROID_SHA256 =",
    raw_mu_k36[
        "centroid_sha256"
    ],
)

print(
    "RAW_K36_TRAIN_ASSIGNMENT_SHA256 =",
    raw_mu_k36[
        "train_assignment_sha256"
    ],
)

print(
    "RAW_K36_VALIDATION_ASSIGNMENT_SHA256 =",
    raw_mu_k36[
        "validation_assignment_sha256"
    ],
)

print()

print(
    "ZSCORE_TRAIN_FEATURE_SHA256 =",
    published_train_zscore_sha,
)

print(
    "ZSCORE_VALIDATION_FEATURE_SHA256 =",
    published_validation_zscore_sha,
)

print()

print(
    "ZSCORE_K22_CENTROID_SHA256 =",
    zscore_k22[
        "centroid_sha256"
    ],
)

print(
    "ZSCORE_K36_CENTROID_SHA256 =",
    zscore_k36[
        "centroid_sha256"
    ],
)

print()

print(
    "ALL_PARTIAL_HASHES_OK =",
    partial_verification[
        "all_hashes_ok"
    ],
)

print(
    "ALL_FINAL_HASHES_OK =",
    final_verification[
        "all_hashes_ok"
    ],
)

print(
    "PARTIAL_DIRECTORY_ABSENT =",
    not KMEANS_PARTIAL_DIR.exists(),
)

print()

print(
    "LABELS_USED =",
    False,
)

print(
    "EXTERNAL_METRICS_COMPUTED =",
    False,
)

print(
    "GRAVITY_SPY_CONTINGENCY_COMPUTED =",
    False,
)

print(
    "SOURCE_EMBEDDINGS_MODIFIED =",
    False,
)

print()

print(
    "NOTEBOOK_05_A5_LABELBLIND_PUBLICATION_PASS =",
    (
        partial_verification[
            "all_hashes_ok"
        ]

        and final_verification[
            "all_hashes_ok"
        ]

        and not KMEANS_PARTIAL_DIR.exists()

        and KMEANS_FINAL_DIR.is_dir()

        and sha256_path(
            KMEANS_FINAL_DIR
            / "kmeans_config_v2.json"
        )
        == EXPECTED_CORRECTED_CONFIG_SHA256
    ),
)


=== NOTEBOOK 05 LABEL-BLIND KMEANS PUBLICATION ===

ARTIFACT_ID = sakai224_full_kmeans_labelblind_v1
FINAL_DIRECTORY = /content/drive/MyDrive/TESI_SAKAI2022/sakai224_full/kmeans/sakai224_full_kmeans_labelblind_v1

FILE_COUNT = 44
TOTAL_BYTES = 18576162

KMEANS_CONFIG_SHA256 = f43a89b318bf22f03a63d2a7e5a995917e97f160c7d8eccbb19fade9a6457d29
SHA256SUMS_SHA256 = 788fee1a5cc07857f2f85b17ed14c0ce392f9be32b822e6d4322dc8cc1e162fc

RAW_K22_CENTROID_SHA256 = 7c196346d32e08d12ebdae94ed602b588827f43b98479eb2dd26fa92ced776f9
RAW_K22_TRAIN_ASSIGNMENT_SHA256 = 13e7f4ed87d3ed459f9e6b7fdf706e1b3758bac735444131e99ad69c92791cf4
RAW_K22_VALIDATION_ASSIGNMENT_SHA256 = f3ca101ee4078814378dee3874d8feb04d89f58c89af95088fb4c3801e39db5f

RAW_K36_CENTROID_SHA256 = 744a2fe7cc32dd1ee358484f39bd0655e2d4cb8e2ff3ee4e6976ea12535dff1f
RAW_K36_TRAIN_ASSIGNMENT_SHA256 = 1f019a4dd7b6dfc04ad73f9fe190b18d4f49bb57e05197638dc2e944bd08c090
RAW_K36_VALIDATION_ASSIGNMENT_SHA256 = 013832aadf859bdfac2069d3d83062ff37ab275fd4021d8

## STEP 05-A6 — PAPER fidelity checkpoint e contratto IIC/spectral

Il blocco k-means del Notebook 05 è una diagnostica PROJECT separata e non
appartiene al classificatore finale proposto da Sakai et al.

La relazione con IIC, over-clustering e spectral clustering è documentata
dal contratto pubblicato della pipeline principale.

### Elementi direttamente PAPER

Il classificatore riceve due versioni dello stesso evento:

$$
x
$$

e

$$
x'=\xi(x),
$$

dove $x$ è l'immagine originale non perturbata e $x'$ è una perturbazione
temporale dello stesso evento.

Entrambe vengono elaborate dallo stesso encoder VAE pre-addestrato con pesi
condivisi:

$$
z=\mu_\phi(x),
$$

$$
z'=\mu_\phi(x').
$$

La dimensione delle feature è:

$$
z,z'\in\mathbb{R}^{512}.
$$

La rete IIC mostrata nella Figura 8 applica alle feature una proiezione
fully-connected con Batch Normalization e ReLU, seguita da un layer
fully-connected e SoftMax.

Il ramo principale produce:

$$
\Phi_C(z)\in\mathbb{R}^{36},
$$

mentre il ramo ausiliario di over-clustering produce:

$$
\Phi_W(z)\in\mathbb{R}^{250}.
$$

Il training massimizza la mutual information fra le predizioni delle due
viste:

$$
I\left(
\Phi(z),
\Phi(z')
\right).
$$

I parametri finali pubblicati sono:

$$
C=36,
$$

$$
W=250,
$$

$$
K=5,
$$

$$
\eta=5\times10^{-4},
$$

$$
\mathrm{batch}=128.
$$

L'ottimizzatore è Adam con:

$$
\beta_1=0.9,
\qquad
\beta_2=0.999,
\qquad
\epsilon=10^{-8}.
$$

I risultati dei $K=5$ classificatori vengono successivamente combinati in
una matrice:

$$
H\in\mathbb{R}^{D\times(KC)}.
$$

Per ogni coppia di eventi viene costruita l'affinità:

$$
A_{ij}
=
\exp
\left(
-\left\|
h^{(i)}-h^{(j)}
\right\|
\right),
$$

seguita da spectral clustering.

---

### Scelte che non devono essere attribuite al PAPER

Il paper non specifica completamente:

- framework software;
- inizializzazione dei layer IIC;
- dettagli numerici della Batch Normalization;
- seed casuali;
- ordine/shuffle esatto degli eventi;
- criterio preciso di arresto dell'IIC;
- implementazione numerica della mutual information;
- peso relativo del termine principale e dell'over-clustering;
- dettagli dell'eigensolver spectral;
- normalizzazione eventualmente applicata prima dello spectral clustering;
- gestione numerica della matrice di affinità.

Questi aspetti non sono attribuiti al PAPER e richiedono una specifica
PROJECT separata, fondata sia su Sakai et al. sia sul lavoro originale
sull'IIC.

---

### Policy delle label

Sakai ha investigato diversi iperparametri e riporta $C=36$ e $K=5$ come
configurazione con prestazione ottimale secondo la propria accuracy.

Nel presente progetto non viene ripetuto questo tuning utilizzando le
Gravity Spy label.

I valori pubblicati:

$$
C=36,\qquad W=250,\qquad K=5
$$

vengono adottati direttamente come configurazione PAPER.

Le label Gravity Spy sono escluse dalla pipeline che produce la partizione
finale:

$$
\mathrm{IIC}
\rightarrow
K=5
\rightarrow
\mathrm{spectral\ clustering}.
$$

La matrice $22\times C$ e le metriche esterne appartengono esclusivamente a
una fase label-aware post-hoc separata.

---

### Nota sulla perturbazione temporale

Il testo del paper descrive uno shift casuale nella direzione temporale nel
range `0–24 px`.

La Figura 7 mostra tuttavia esempi sia negativi sia positivi rispetto al crop
centrale, inclusi `-24 px` e `+24 px`.

Questa ambiguità viene mantenuta esplicitamente aperta.

La specifica PROJECT del training IIC adotta una sola interpretazione, scelta
senza utilizzare le label per distinguere fra le alternative.

---

### Stato al termine del Notebook 05

Il Notebook 05 chiude esclusivamente la diagnostica k-means delle feature
VAE.

Sono congelati:

- `raw_mu`, analisi primaria;
- `zscore_train_only`, sensitivity analysis;
- $k=22$ e $k=36$;
- centroidi train;
- assignment train e validation;
- diagnostiche interne;
- SHA-256;
- artefatto label-blind persistente.

Non sono state calcolate:

- matrici di contingenza Gravity Spy;
- purity;
- NMI rispetto alle label;
- ARI rispetto alle label;
- homogeneity;
- completeness.

La pipeline IIC con over-clustering e spectral clustering è trattata
separatamente nel Notebook 06. La baseline k-means del Notebook 05 non
sostituisce l'IIC e non è direttamente confrontabile con l'accuracy del
90.9% riportata da Sakai et al.

In [10]:
# ============================================================
# NOTEBOOK 05 — FINAL LABEL-BLIND RESTORE / HANDOFF AUDIT
# STEP 05-A6
#
# READ ONLY.
# NO labels.
# NO clustering.
# NO writes.
# ============================================================

from pathlib import Path

import csv
import json


# ============================================================
# PERSISTENT ARTIFACT
# ============================================================

assert KMEANS_FINAL_DIR.is_dir()
assert not KMEANS_PARTIAL_DIR.exists()


# ============================================================
# RE-READ METADATA FROM DRIVE
# ============================================================

with (
    KMEANS_FINAL_DIR
    / "metadata.json"
).open(
    "r",
    encoding="utf-8",
) as handle:

    restored_kmeans_metadata = json.load(
        handle
    )


metadata_policy_gate = (
    restored_kmeans_metadata[
        "artifact_id"
    ]
    == "sakai224_full_kmeans_labelblind_v1"

    and restored_kmeans_metadata[
        "main_scientific_goal"
    ]
    == "sakai224_full_vs_Sakai"

    and restored_kmeans_metadata[
        "labels_used"
    ]
    is False

    and restored_kmeans_metadata[
        "external_metrics_computed"
    ]
    is False

    and restored_kmeans_metadata[
        "gravity_spy_contingency_computed"
    ]
    is False

    and restored_kmeans_metadata[
        "external_evaluation_deferred_until_after_IIC_spectral"
    ]
    is True
)


assert metadata_policy_gate


# ============================================================
# FULL PACKAGE CHECKSUM AUDIT
# ============================================================

restored_verification = verify_package(
    KMEANS_FINAL_DIR
)


package_gate = (
    restored_verification[
        "all_hashes_ok"
    ]

    and len(
        restored_verification[
            "temp_files"
        ]
    ) == 0

    and sha256_path(
        KMEANS_FINAL_DIR
        / "SHA256SUMS.txt"
    )
    == SHA256SUMS_SHA256
)


assert package_gate


# ============================================================
# LABEL-FREE ROW-MAP AUDIT
# ============================================================

def read_header_only(
    path,
):

    with Path(path).open(
        "r",
        newline="",
        encoding="utf-8",
    ) as handle:

        reader = csv.reader(
            handle
        )

        return next(
            reader
        )


restored_train_header = read_header_only(
    KMEANS_FINAL_DIR
    / TRAIN_ROW_MAP_NAME
)


restored_validation_header = read_header_only(
    KMEANS_FINAL_DIR
    / VALIDATION_ROW_MAP_NAME
)


row_map_label_gate = (
    restored_train_header
    == [
        "split",
        "split_pos",
        "manifest_index",
        "event_index",
        "event_id",
    ]

    and restored_validation_header
    == [
        "split",
        "split_pos",
        "manifest_index",
        "event_index",
        "event_id",
    ]

    and "label"
    not in restored_train_header

    and "label"
    not in restored_validation_header
)


assert row_map_label_gate


# ============================================================
# SOURCE EMBEDDINGS STILL FROZEN
# ============================================================

source_embedding_gate = (
    sha256_file(
        TRAIN_BIN_PATH
    )
    == EXPECTED_TRAIN_MU_SHA256

    and sha256_file(
        VALIDATION_BIN_PATH
    )
    == EXPECTED_VALIDATION_MU_SHA256
)


assert source_embedding_gate


# ============================================================
# PAPER-FIDELITY HANDOFF CONTRACT
#
# This does NOT attempt to encode unpublished details.
# ============================================================

IIC_PAPER_HANDOFF = {

    "vae_encoder_checkpoint":
        "FINAL epoch 100",

    "feature":
        "mu",

    "latent_dim":
        512,

    "original_view":
        "central_non_perturbed",

    "perturbed_view":
        "time_direction_only",

    "output_classes_C":
        36,

    "overcluster_classes_W":
        250,

    "classifier_count_K":
        5,

    "learning_rate":
        5.0e-4,

    "batch_size":
        128,

    "adam_beta1":
        0.9,

    "adam_beta2":
        0.999,

    "adam_epsilon":
        1.0e-8,

    "main_objective":
        "maximize_mutual_information",

    "overclustering":
        True,

    "spectral_h_shape":
        "D x (K*C)",

    "spectral_affinity":
        "exp(-||h_i-h_j||)",

    "labels_for_training":
        False,

    "gravity_spy_evaluation":
        "deferred_until_final_partition",

    "unresolved_before_implementation": [

        "IIC initialization",

        "IIC exact epoch/stopping policy",

        "BatchNorm numerical semantics",

        "exact MI numerical estimator",

        "main/overcluster loss combination",

        "temporal shift text-vs-figure ambiguity",

        "spectral clustering numerical details",
    ],
}


handoff_gate = (
    IIC_PAPER_HANDOFF[
        "latent_dim"
    ] == 512

    and IIC_PAPER_HANDOFF[
        "output_classes_C"
    ] == 36

    and IIC_PAPER_HANDOFF[
        "overcluster_classes_W"
    ] == 250

    and IIC_PAPER_HANDOFF[
        "classifier_count_K"
    ] == 5

    and IIC_PAPER_HANDOFF[
        "learning_rate"
    ] == 5.0e-4

    and IIC_PAPER_HANDOFF[
        "batch_size"
    ] == 128

    and IIC_PAPER_HANDOFF[
        "labels_for_training"
    ] is False
)


assert handoff_gate


# ============================================================
# FINAL REPORT
# ============================================================

print()
print(
    "=== NOTEBOOK 05 FINAL FIDELITY / HANDOFF AUDIT ==="
)

print()

print(
    "KMEANS_ARTIFACT =",
    KMEANS_FINAL_DIR,
)

print(
    "KMEANS_SHA256SUMS_SHA256 =",
    SHA256SUMS_SHA256,
)

print()

print(
    "METADATA_POLICY_GATE =",
    metadata_policy_gate,
)

print(
    "PACKAGE_GATE =",
    package_gate,
)

print(
    "ROW_MAP_LABEL_GATE =",
    row_map_label_gate,
)

print(
    "SOURCE_EMBEDDING_GATE =",
    source_embedding_gate,
)

print(
    "IIC_PAPER_HANDOFF_GATE =",
    handoff_gate,
)

print()

print(
    "IIC_C =",
    IIC_PAPER_HANDOFF[
        "output_classes_C"
    ],
)

print(
    "IIC_W =",
    IIC_PAPER_HANDOFF[
        "overcluster_classes_W"
    ],
)

print(
    "IIC_K =",
    IIC_PAPER_HANDOFF[
        "classifier_count_K"
    ],
)

print(
    "IIC_LR =",
    IIC_PAPER_HANDOFF[
        "learning_rate"
    ],
)

print(
    "IIC_BATCH =",
    IIC_PAPER_HANDOFF[
        "batch_size"
    ],
)

print()

print(
    "LABELS_READ =",
    False,
)

print(
    "EXTERNAL_METRICS_COMPUTED =",
    False,
)

print(
    "GRAVITY_SPY_CONTINGENCY_COMPUTED =",
    False,
)

print(
    "OFFICIAL_ARTIFACTS_MODIFIED_IN_A6 =",
    False,
)

print()

print(
    "NOTEBOOK_05_FORMALLY_CLOSED_LABELBLIND =",
    (
        metadata_policy_gate
        and package_gate
        and row_map_label_gate
        and source_embedding_gate
        and handoff_gate
    ),
)


=== NOTEBOOK 05 FINAL FIDELITY / HANDOFF AUDIT ===

KMEANS_ARTIFACT = /content/drive/MyDrive/TESI_SAKAI2022/sakai224_full/kmeans/sakai224_full_kmeans_labelblind_v1
KMEANS_SHA256SUMS_SHA256 = 788fee1a5cc07857f2f85b17ed14c0ce392f9be32b822e6d4322dc8cc1e162fc

METADATA_POLICY_GATE = True
PACKAGE_GATE = True
ROW_MAP_LABEL_GATE = True
SOURCE_EMBEDDING_GATE = True
IIC_PAPER_HANDOFF_GATE = True

IIC_C = 36
IIC_W = 250
IIC_K = 5
IIC_LR = 0.0005
IIC_BATCH = 128

LABELS_READ = False
EXTERNAL_METRICS_COMPUTED = False
GRAVITY_SPY_CONTINGENCY_COMPUTED = False
OFFICIAL_ARTIFACTS_MODIFIED_IN_A6 = False

NOTEBOOK_05_FORMALLY_CLOSED_LABELBLIND = True
